# MAG7 Article-Event Label Classifier - Interaction Model

This notebook fuses article-event labels (`article_event_labels.parquet`) with market context and interaction features to classify each MAG7 news event as bullish, bearish, or neutral, using a softmax neural classifier trained jointly with a Logic Tensor Network (LTN) rule layer. Each row is one ticker-specific article-event label; forward returns are computed directly from prices. Sections 1-12 build and evaluate the primary model on a single chronological split; Sections 13-15 repeat the same evaluation on a second, differently-timed split to test how stable the results are; Section 16 synthesises both and states plainly what the combined evidence does and does not support.

## 1. Aim And Setup

This section imports libraries, sets random seeds, defines the MAG7 return label configuration, and declares the chronological training, validation and test windows.

In [1]:
from __future__ import annotations

import json
from itertools import combinations
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import binomtest

import matplotlib.pyplot as plt
import seaborn as sns

import ltn

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 260)

WORK_DIR = Path.cwd()

# Configure these if files are not in the working folder. The notebook also
# searches common output folders, nested folders, and Downloads.
DATA_DIR = Path(globals().get("DATA_DIR", WORK_DIR))
ARTICLE_EVENTS_FILE = globals().get("ARTICLE_EVENTS_FILE", "article_event_labels.parquet")
MAG7_PRICE_FILE = globals().get("MAG7_PRICE_FILE", "mag7_daily_prices.parquet")
SP500_PRICE_FILE = globals().get("SP500_PRICE_FILE", "sp500_daily_prices.parquet")


def find_existing_file(name_or_path, search_roots):
    path = Path(name_or_path)
    if path.is_absolute() and path.exists():
        return path

    candidates = []
    for root in search_roots:
        root = Path(root)
        candidates.extend([
            root / path,
            root / "outputs" / "mag7_prompt_v7_full" / path,
            root / "outputs" / path,
        ])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    for root in search_roots:
        root = Path(root)
        if root.exists():
            matches = sorted(root.rglob(path.name))
            if matches:
                return matches[0]
            numbered_matches = sorted(root.rglob(f"{path.stem} (*){path.suffix}"))
            if numbered_matches:
                return numbered_matches[-1]

    raise FileNotFoundError(
        f"Could not find {name_or_path!r}. Set DATA_DIR or explicit file variables before running this cell."
    )


SEARCH_ROOTS = list(dict.fromkeys([DATA_DIR, WORK_DIR, WORK_DIR.parent, Path.home() / "Downloads"]))

ARTICLE_EVENTS_PATH = find_existing_file(ARTICLE_EVENTS_FILE, SEARCH_ROOTS)
MAG7_PRICE_PATH = find_existing_file(MAG7_PRICE_FILE, SEARCH_ROOTS)
SP500_PRICE_PATH = find_existing_file(SP500_PRICE_FILE, SEARCH_ROOTS)

OUTPUT_DIR = WORK_DIR / "FINAL_17_mag7_article_event_label_interaction_clean_no_robust"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_HORIZON = 3
HORIZON_CANDIDATES = [1, 2, 3, 4, 5]
RETURN_THRESHOLD = 0.005

TRAINING = ("2023-01-01", "2025-12-31")
VALIDATION = ("2026-01-01", "2026-02-28")
TEST = ("2026-03-01", "2026-06-01")

EPOCHS = 500
LR = 0.01
LOGIC_WEIGHT = 0.30
SEED = 7
ROBUST_SEEDS = [1, 2, 3, 4, 5, 7, 11, 13, 17, 19]

LABELS = ["bearish", "bullish", "neutral"]
ANTECEDENT_THRESHOLD = 0.50
MIN_TRAINING_EVENTS = 6
MIN_VALIDATION_EVENTS = 5
MIN_TRAINING_ACCURACY = 60.0
MIN_VALIDATION_ACCURACY = 60.0

PRIMARY_VARIANT = "full_ltn_qwen_context_interaction"
HORIZON_AUDIT_VARIANT = PRIMARY_VARIANT
THRESHOLD_AUDIT_VARIANT = PRIMARY_VARIANT
TUNE_VARIANT = PRIMARY_VARIANT
TRUE_SELECTIVE_FILTER_VARIANT = PRIMARY_VARIANT

SAVE_OUTPUTS = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("working folder:", WORK_DIR)
print("article event labels:", ARTICLE_EVENTS_PATH)
print("mag7 prices:", MAG7_PRICE_PATH)
print("sp500 prices:", SP500_PRICE_PATH)
print("output:", OUTPUT_DIR)
print("primary variant:", PRIMARY_VARIANT)
print("ltn:", getattr(ltn, "__version__", "unknown"))
print("torch:", torch.__version__, "device:", device)

working folder: /home/jovyan/Stock-Sentiment-Prediction
article event labels: /home/jovyan/Stock-Sentiment-Prediction/outputs/mag7_prompt_v7_full/article_event_labels.parquet
mag7 prices: /home/jovyan/Stock-Sentiment-Prediction/mag7_daily_prices.parquet
sp500 prices: /home/jovyan/Stock-Sentiment-Prediction/sp500_daily_prices.parquet
output: /home/jovyan/Stock-Sentiment-Prediction/FINAL_17_mag7_article_event_label_interaction_clean_no_robust
primary variant: full_ltn_qwen_context_interaction
ltn: unknown
torch: 2.12.1+cu126 device: cuda


In [2]:
def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

reset_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print("Could not force all deterministic algorithms:", e)

## 2. Load Article-Event Label And Price Data

This section loads `article_event_labels.parquet` plus MAG7 and S&P 500 prices. It keeps one row per ticker-article label, so the setup is event-conditioned and closer to the original notebook than the daily-packet panel version.


In [3]:
events_all = pd.read_parquet(ARTICLE_EVENTS_PATH).copy()
prices = pd.read_parquet(MAG7_PRICE_PATH).copy()
sp500 = pd.read_parquet(SP500_PRICE_PATH).copy()

events_all["time_published"] = pd.to_datetime(
    events_all["time_published"],
    utc=True,
    errors="coerce",
)
events_all["event_date"] = pd.to_datetime(events_all["event_date"]).dt.tz_localize(None)
prices["date"] = pd.to_datetime(prices["date"]).dt.tz_localize(None)
sp500["date"] = pd.to_datetime(sp500["date"]).dt.tz_localize(None)

# One modelling row per ticker-article event label. This is closer to the old
# event-conditioned notebook than daily packets, but uses the new extraction
# output instead of old event episodes/refinement rows.
events = (
    events_all
    .drop_duplicates(["ticker", "article_uid"], keep="last")
    .sort_values(["ticker", "time_published", "article_uid"])
    .reset_index(drop=True)
)

if "article_event_label_id" not in events.columns:
    events["article_event_label_id"] = (
        events["ticker"].astype(str)
        + "_"
        + events["article_uid"].astype(str)
    )

events["event_id"] = events["article_event_label_id"].astype(str)
events["has_article_event"] = 1.0
events["representative_title"] = events.get("representative_title", "").fillna("")
events["event_type"] = events.get("event_type", "missing").fillna("missing").astype(str)
events["direction"] = events.get("direction", "none").fillna("none").astype(str)
events["confidence"] = events.get("confidence", "missing").fillna("missing").astype(str)
events["company_relevance"] = events.get("company_relevance", "missing").fillna("missing").astype(str)
events["mechanism"] = events.get("mechanism", "").fillna("")
events["event_key"] = events.get("event_key", "").fillna("")

# Same-day coverage controls. These do not deduplicate the modelling rows, but
# they let the model/rules see whether a row belongs to a heavily repeated
# ticker-day/event-type/direction cluster.
same_day = events.groupby(["ticker", "event_date"])["article_uid"].transform("count")
same_type_dir = events.groupby(
    ["ticker", "event_date", "event_type", "direction"]
)["article_uid"].transform("count")

events["same_day_article_event_count"] = same_day.astype(float)
events["same_day_event_type_direction_count"] = same_type_dir.astype(float)
events["article_event_weight"] = (
    1.0 / events["same_day_event_type_direction_count"].clip(lower=1)
)


def pick_price_column(price_df: pd.DataFrame, candidates: list[str], label: str) -> str:
    for col in candidates:
        if col in price_df.columns:
            return col

    raise RuntimeError(
        f"Could not find a usable {label} column. "
        f"Tried {candidates}. Available columns: {list(price_df.columns)}"
    )


def build_open_to_close_return_table(
    price_df: pd.DataFrame,
    horizons,
) -> pd.DataFrame:
    """
    Event-date open to horizon close return.

    The event_date has already been mapped so that article-event features are
    known before the target trading session opens.

    Horizon meaning:
    - 1 = event_date open -> event_date close
    - 2 = event_date open -> next trading-day close
    - 3 = event_date open -> close two trading days later
    - etc.
    """
    open_col = pick_price_column(
        price_df,
        ["adj_open", "open", "Open", "open_price"],
        "open price",
    )

    close_col = pick_price_column(
        price_df,
        ["adj_close", "close", "Close", "close_price"],
        "close price",
    )

    frames = []

    for horizon in sorted({int(h) for h in horizons if pd.notna(h)}):
        if horizon < 1:
            raise ValueError("Open-to-horizon-close horizons must be >= 1.")

        for ticker, group in price_df.groupby("ticker"):
            g = group.sort_values("date").copy()

            g["event_date"] = g["date"]
            g["event_open"] = pd.to_numeric(g[open_col], errors="coerce")
            g["event_close"] = pd.to_numeric(g[close_col], errors="coerce").shift(
                -(horizon - 1)
            )
            g["horizon_days"] = horizon
            g["simple_return"] = g["event_close"] / g["event_open"] - 1.0

            frames.append(
                g[
                    [
                        "ticker",
                        "event_date",
                        "horizon_days",
                        "simple_return",
                        "event_open",
                        "event_close",
                    ]
                ]
            )

    return pd.concat(frames, ignore_index=True)


def realised_label_from_return(value: float, threshold: float = RETURN_THRESHOLD) -> str:
    if pd.isna(value):
        return "neutral"
    if value > threshold:
        return "bullish"
    if value < -threshold:
        return "bearish"
    return "neutral"


all_return_horizons = sorted(set(HORIZON_CANDIDATES) | {PRIMARY_HORIZON})
simple_returns = build_open_to_close_return_table(
    prices,
    all_return_horizons,
)

print("article-event rows:", len(events))
print("event_date range:", events["event_date"].min(), "to", events["event_date"].max())

display(
    events
    .groupby(events["event_date"].dt.year)
    .size()
    .rename("article_event_labels")
    .to_frame()
)

display(
    events[
        [
            "ticker",
            "time_published",
            "event_date",
            "event_type",
            "direction",
            "confidence",
            "same_day_article_event_count",
            "same_day_event_type_direction_count",
            "representative_title",
        ]
    ].head()
)


article-event rows: 12272
event_date range: 2023-01-03 00:00:00 to 2026-06-01 00:00:00


,article_event_labels
event_date,
2023,467
2024,589
2025,6293
2026,4923


,ticker,time_published,event_date,event_type,direction,confidence,same_day_article_event_count,same_day_event_type_direction_count,representative_title
0,AAPL,2023-01-04 11:52:00+00:00,2023-01-04,major_customer_contract,positive,high,1.0,1.0,Apple to sign Luxshare for iPhone production i...
1,AAPL,2023-01-05 03:59:02+00:00,2023-01-05,novel_product_event,positive,high,1.0,1.0,Apple brings AI narration to audiobooks
2,AAPL,2023-01-06 04:31:08+00:00,2023-01-06,supply_chain_production_disruption,negative,high,1.0,1.0,Apple Analyst Says BOE Will Become Largest Dis...
3,AAPL,2023-01-11 01:39:00+00:00,2023-01-11,novel_product_event,positive,high,3.0,1.0,Apple to start using in-house screens from 202...
4,AAPL,2023-01-11 04:03:16+00:00,2023-01-11,demand_supply_revision,negative,high,3.0,1.0,Inari pares loss as analysts see limited impac...


## 3. Return Horizon Configuration

The final MAG7 label uses a five-day open-to-close return and a +/-0.5% threshold. The validation-only horizon audit below (Section 7) ranks the 5-day horizon highest on both accuracy (49.5%, versus 47.0% for the next-best 4-day horizon) and aligned mean return (0.385%), and it is the only horizon whose aligned return clears conventional significance under the cluster-bootstrap test (p = 0.026). It is therefore used as the primary horizon (Section 8). Candidate horizons are kept available for that audit.

## 4. Build Horizon-Independent Article-Event Features

This section constructs article-event categorical features, market-context features and interaction features before attaching the final return label. Same-day article coverage controls are log-scaled.


In [4]:
def add_price_context(price_df: pd.DataFrame, prefix: str, group_col: str | None = None) -> pd.DataFrame:
    out = price_df.sort_values(([group_col] if group_col else []) + ["date"]).copy()
    grouped = out.groupby(group_col, group_keys=False) if group_col else [(None, out)]
    parts = []

    for _, g in grouped:
        g = g.sort_values("date").copy()
        px = pd.to_numeric(g["adj_close"], errors="coerce")
        daily = px.pct_change()

        for window in [1, 3, 5, 20]:
            g[f"{prefix}_ret_{window}d_pre"] = px.pct_change(window).shift(1)

        g[f"{prefix}_abs_ret_1d_pre"] = daily.abs().shift(1)
        g[f"{prefix}_vol_5d_pre"] = daily.rolling(5).std().shift(1)
        g[f"{prefix}_vol_20d_pre"] = daily.rolling(20).std().shift(1)
        parts.append(g)

    return pd.concat(parts, ignore_index=True)


ticker_ctx = add_price_context(prices, "ticker", "ticker")
sp500_ctx = add_price_context(sp500, "sp500", None).drop(columns=["ticker"], errors="ignore")

ctx = events.merge(
    ticker_ctx[[
        "ticker", "date", "ticker_ret_1d_pre", "ticker_ret_3d_pre", "ticker_ret_5d_pre", "ticker_ret_20d_pre",
        "ticker_abs_ret_1d_pre", "ticker_vol_5d_pre", "ticker_vol_20d_pre",
    ]],
    left_on=["ticker", "event_date"],
    right_on=["ticker", "date"],
    how="left",
).drop(columns=["date"], errors="ignore")

ctx = ctx.merge(
    sp500_ctx[[
        "date", "sp500_ret_1d_pre", "sp500_ret_3d_pre", "sp500_ret_5d_pre", "sp500_ret_20d_pre",
        "sp500_abs_ret_1d_pre", "sp500_vol_5d_pre", "sp500_vol_20d_pre",
    ]],
    left_on="event_date",
    right_on="date",
    how="left",
).drop(columns=["date"], errors="ignore")

for h in ["1d", "3d", "5d", "20d"]:
    ctx[f"relative_ret_{h}_pre"] = ctx[f"ticker_ret_{h}_pre"] - ctx[f"sp500_ret_{h}_pre"]

ctx = ctx.sort_values(["ticker", "time_published", "article_uid"]).reset_index(drop=True)
ctx["event_density_ticker_7d"] = 0.0
ctx["event_density_ticker_30d"] = 0.0
ctx["similar_density_ticker_30d"] = 0.0

for ticker, group in ctx.groupby("ticker"):
    for idx, ts, event_type, direction in zip(
        group.index,
        group["time_published"],
        group["event_type"],
        group["direction"],
    ):
        prev = group.loc[group["time_published"].lt(ts)]
        last7 = prev.loc[prev["time_published"].ge(ts - pd.Timedelta(days=7))]
        last30 = prev.loc[prev["time_published"].ge(ts - pd.Timedelta(days=30))]
        similar30 = last30.loc[
            last30["event_type"].eq(event_type)
            | last30["direction"].eq(direction)
        ]

        ctx.loc[idx, "event_density_ticker_7d"] = len(last7)
        ctx.loc[idx, "event_density_ticker_30d"] = len(last30)
        ctx.loc[idx, "similar_density_ticker_30d"] = len(similar30)

hour = ctx["time_published"].dt.hour + ctx["time_published"].dt.minute / 60
ctx["hour_sin_01"] = (np.sin(2 * np.pi * hour / 24) + 1) / 2
ctx["hour_cos_01"] = (np.cos(2 * np.pi * hour / 24) + 1) / 2
ctx["is_monday"] = (ctx["event_date"].dt.weekday == 0).astype(float)
ctx["is_friday"] = (ctx["event_date"].dt.weekday == 4).astype(float)

display(ctx[[
    "ticker", "time_published", "event_date", "event_type", "direction", "confidence",
    "relative_ret_5d_pre", "sp500_vol_5d_pre", "ticker_vol_5d_pre",
    "event_density_ticker_7d", "event_density_ticker_30d", "similar_density_ticker_30d",
]].head(10))

,ticker,time_published,event_date,event_type,direction,confidence,relative_ret_5d_pre,sp500_vol_5d_pre,ticker_vol_5d_pre,event_density_ticker_7d,event_density_ticker_30d,similar_density_ticker_30d
0,AAPL,2023-01-04 11:52:00+00:00,2023-01-04,major_customer_contract,positive,high,NaN,NaN,NaN,0.0,0.0,0.0
1,AAPL,2023-01-05 03:59:02+00:00,2023-01-05,novel_product_event,positive,high,NaN,NaN,NaN,1.0,1.0,1.0
2,AAPL,2023-01-06 04:31:08+00:00,2023-01-06,supply_chain_production_disruption,negative,high,NaN,NaN,NaN,2.0,2.0,0.0
3,AAPL,2023-01-11 01:39:00+00:00,2023-01-11,novel_product_event,positive,high,0.020384,0.012639,0.017345,3.0,3.0,2.0
4,AAPL,2023-01-11 04:03:16+00:00,2023-01-11,demand_supply_revision,negative,high,0.020384,0.012639,0.017345,4.0,4.0,1.0
5,AAPL,2023-01-11 04:31:49+00:00,2023-01-11,regulatory_legal_outcome,negative,high,0.020384,0.012639,0.017345,5.0,5.0,2.0
6,AAPL,2023-01-13 03:59:02+00:00,2023-01-13,management_leadership_change,negative,high,0.021136,0.009179,0.015566,4.0,6.0,3.0
7,AAPL,2023-01-13 03:59:02+00:00,2023-01-13,management_leadership_change,negative,high,0.021136,0.009179,0.015566,4.0,6.0,3.0
8,AAPL,2023-01-13 11:45:00+00:00,2023-01-13,strategy_capital_allocation,positive,high,0.021136,0.009179,0.015566,5.0,8.0,3.0
9,AAPL,2023-01-19 13:12:00+00:00,2023-01-19,supply_chain_production_disruption,negative,high,0.031817,0.010460,0.010278,3.0,9.0,5.0


## 5. Chronological Split And Feature Specifications

This section assigns each event to the training, validation or test period and defines the feature groups used by the model variants.

In [5]:
def assign_period(d):
    d = pd.Timestamp(d)
    if pd.Timestamp(TRAINING[0]) <= d <= pd.Timestamp(TRAINING[1]):
        return "training"
    if pd.Timestamp(VALIDATION[0]) <= d <= pd.Timestamp(VALIDATION[1]):
        return "validation"
    if pd.Timestamp(TEST[0]) <= d <= pd.Timestamp(TEST[1]):
        return "test"
    return "outside"


ctx["period"] = ctx["event_date"].map(assign_period)
fit_mask = ctx["period"].eq("training")


def fit_abs_scale(col, q=0.80):
    s = pd.to_numeric(ctx.loc[fit_mask, col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    scale = s.abs().quantile(q)
    return float(scale) if np.isfinite(scale) and scale != 0 else 1.0


def fit_quantile(col, q, fallback=1.0):
    s = pd.to_numeric(ctx.loc[fit_mask, col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    value = s.quantile(q)
    return float(value) if np.isfinite(value) and value != 0 else fallback


def soft_pos_with_scale(s, scale):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (s.clip(lower=0) / scale).clip(0, 1).fillna(0)


def soft_neg_with_scale(s, scale):
    return soft_pos_with_scale(-pd.to_numeric(s, errors="coerce"), scale)


def soft_low_with_cutoff(s, cutoff):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (1 - (s / cutoff)).clip(0, 1).fillna(0)


def soft_high_with_cutoff(s, cutoff):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (s / cutoff - 1).clip(0, 1).fillna(0)


ticker_5d_scale = fit_abs_scale("ticker_ret_5d_pre")
market_5d_scale = fit_abs_scale("sp500_ret_5d_pre")
relative_5d_scale = fit_abs_scale("relative_ret_5d_pre")

ctx["ticker_prior_up_5d"] = soft_pos_with_scale(ctx["ticker_ret_5d_pre"], ticker_5d_scale)
ctx["ticker_prior_down_5d"] = soft_neg_with_scale(ctx["ticker_ret_5d_pre"], ticker_5d_scale)
ctx["market_prior_up_5d"] = soft_pos_with_scale(ctx["sp500_ret_5d_pre"], market_5d_scale)
ctx["market_prior_down_5d"] = soft_neg_with_scale(ctx["sp500_ret_5d_pre"], market_5d_scale)
ctx["relative_prior_up_5d"] = soft_pos_with_scale(ctx["relative_ret_5d_pre"], relative_5d_scale)
ctx["relative_prior_down_5d"] = soft_neg_with_scale(ctx["relative_ret_5d_pre"], relative_5d_scale)
ctx["quiet_market_5d"] = soft_low_with_cutoff(ctx["sp500_vol_5d_pre"], fit_quantile("sp500_vol_5d_pre", 0.40))
ctx["volatile_market_5d"] = soft_high_with_cutoff(ctx["sp500_vol_5d_pre"], fit_quantile("sp500_vol_5d_pre", 0.70))
ctx["quiet_ticker_5d"] = soft_low_with_cutoff(ctx["ticker_vol_5d_pre"], fit_quantile("ticker_vol_5d_pre", 0.40))
ctx["volatile_ticker_5d"] = soft_high_with_cutoff(ctx["ticker_vol_5d_pre"], fit_quantile("ticker_vol_5d_pre", 0.70))

for col in ["event_density_ticker_7d", "event_density_ticker_30d", "similar_density_ticker_30d"]:
    denom = max(ctx.loc[fit_mask, col].quantile(0.95), 1)
    ctx[col + "_01"] = (ctx[col] / denom).clip(0, 1)

ctx["novel_context_30d"] = (1 - ctx["similar_density_ticker_30d_01"]).clip(0, 1)

dummy_parts = []
dummy_cols = []
categorical_cols = ["event_type", "direction", "confidence", "company_relevance"]

for col in categorical_cols:
    categories = sorted(set(ctx.loc[fit_mask, col].fillna("missing").astype(str)))
    values = ctx[col].fillna("missing").astype(str)
    for category in categories:
        dummy_name = f"{col}_{category}"
        dummy_parts.append(values.eq(category).astype(float).rename(dummy_name))
        dummy_cols.append(dummy_name)

dummies = pd.concat(dummy_parts, axis=1) if dummy_parts else pd.DataFrame(index=ctx.index)
ctx = pd.concat([ctx, dummies], axis=1)

count_feature_cols = []
for col in ["same_day_article_event_count", "same_day_event_type_direction_count"]:
    feature_col = f"log1p_{col}"
    ctx[feature_col] = np.log1p(pd.to_numeric(ctx[col], errors="coerce").fillna(0))
    denom = max(ctx.loc[fit_mask, feature_col].quantile(0.95), 1e-9)
    ctx[feature_col] = (ctx[feature_col] / denom).clip(0, 1)
    count_feature_cols.append(feature_col)

ctx["article_event_weight_01"] = pd.to_numeric(ctx["article_event_weight"], errors="coerce").fillna(0).clip(0, 1)

context_cols = [
    "ticker_prior_up_5d", "ticker_prior_down_5d", "market_prior_up_5d", "market_prior_down_5d",
    "relative_prior_up_5d", "relative_prior_down_5d", "quiet_market_5d", "volatile_market_5d",
    "quiet_ticker_5d", "volatile_ticker_5d",
    "event_density_ticker_7d_01", "event_density_ticker_30d_01", "similar_density_ticker_30d_01",
    "novel_context_30d", "hour_sin_01", "hour_cos_01", "is_monday", "is_friday",
]

# Keep old variant keys for downstream compatibility. In this notebook,
# qwen_cols means article-event label features from the first LLM extraction,
# not the old second-stage refinement fields.
qwen_cols = list(dummies.columns) + count_feature_cols + ["article_event_weight_01", "has_article_event"]

interaction_qwen_cols = [
    c for c in dummy_cols
    if c.startswith("event_type_")
    or c.startswith("direction_")
]

interaction_context_cols = [
    "relative_prior_up_5d", "relative_prior_down_5d", "quiet_market_5d", "volatile_market_5d",
    "quiet_ticker_5d", "volatile_ticker_5d", "event_density_ticker_7d_01", "novel_context_30d",
]

interaction_data = {}
for q in interaction_qwen_cols:
    for c in interaction_context_cols:
        name = f"interaction__{q}__x__{c}"
        interaction_data[name] = ctx[q].fillna(0) * ctx[c].fillna(0)

interaction_cols = list(interaction_data.keys())
if interaction_data:
    ctx = pd.concat([ctx, pd.DataFrame(interaction_data, index=ctx.index)], axis=1)

feature_df = ctx.loc[ctx["period"].isin(["training", "validation", "test"])].copy()

for col in qwen_cols + context_cols + interaction_cols:
    feature_df[col] = pd.to_numeric(feature_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 1)

rule_block_feature_cols = {
    "qwen_semantic": list(qwen_cols),
    "market_context": list(context_cols),
    "qwen_context_interaction": list(interaction_cols),
}

variant_specs = {
    "full_ltn_qwen_only": {"feature_cols": qwen_cols, "rule_blocks": ["qwen_semantic"]},
    "full_ltn_context_only": {"feature_cols": context_cols, "rule_blocks": ["market_context"]},
    "full_ltn_qwen_plus_context": {"feature_cols": qwen_cols + context_cols, "rule_blocks": ["qwen_semantic", "market_context"]},
    "full_ltn_qwen_context_interaction": {"feature_cols": qwen_cols + context_cols + interaction_cols, "rule_blocks": ["qwen_semantic", "market_context", "qwen_context_interaction"]},
}

for spec in variant_specs.values():
    spec["feature_cols"] = list(dict.fromkeys(spec["feature_cols"]))
    spec["return_col"] = "simple_return"

feature_manifest = pd.DataFrame([
    {
        "variant": name,
        "n_features": len(spec["feature_cols"]),
        "rule_blocks": ", ".join(spec["rule_blocks"]),
        "is_primary_variant": name == PRIMARY_VARIANT,
        "features": ", ".join(spec["feature_cols"]),
    }
    for name, spec in variant_specs.items()
])

display(feature_df["period"].value_counts().rename("article_event_labels").to_frame())
display(feature_manifest)

if SAVE_OUTPUTS:
    feature_manifest.to_csv(OUTPUT_DIR / "full_ltn_feature_manifest.csv", index=False)

,article_event_labels
period,
training,7349
test,2957
validation,1966


,variant,n_features,rule_blocks,is_primary_variant,features
0,full_ltn_qwen_only,22,qwen_semantic,False,"event_type_cybersecurity_privacy_incident, eve..."
1,full_ltn_context_only,18,market_context,False,"ticker_prior_up_5d, ticker_prior_down_5d, mark..."
2,full_ltn_qwen_plus_context,40,"qwen_semantic, market_context",False,"event_type_cybersecurity_privacy_incident, eve..."
3,full_ltn_qwen_context_interaction,160,"qwen_semantic, market_context, qwen_context_in...",True,"event_type_cybersecurity_privacy_incident, eve..."


## 6. Model, Rule-Mining And Evaluation Helpers

This section defines the softmax neural classifier, LTN predicates, rule-mining utilities, training loop, scoring helper and summary metrics.

In [6]:
class OutcomeSoftmaxMLP(nn.Module):
    def __init__(self, input_dim, n_classes=3):
        super().__init__()
        hidden = max(12, min(80, input_dim * 2))
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        return self.net(x)


class ClassPredicate(nn.Module):
    """
        Each LTN predicate is now one class probability from a shared softmax classifier.
    This preserves fuzzy truth values, but avoids three independent sigmoid outputs.
    """
    def __init__(self, classifier, class_index):
        super().__init__()
        self.classifier = classifier
        self.class_index = class_index

    def forward(self, x):
        logits = self.classifier(x)
        probs = torch.softmax(logits, dim=1)
        return probs[:, self.class_index].reshape(-1)


def truth_value(obj):
    return obj.value if hasattr(obj, "value") else obj


def feature_tensor(data, cols):
    x = (
        data[cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .clip(0, 1)
        .to_numpy(dtype=np.float32)
    )
    return torch.tensor(x, dtype=torch.float32, device=device)


def target_tensor(data):
    """
        Cross-entropy expects integer class labels, not one-hot MSE targets.
    """
    label_to_idx = {label: i for i, label in enumerate(LABELS)}
    y = data["target_label"].map(label_to_idx).to_numpy()
    return torch.tensor(y, dtype=torch.long, device=device)


def label_return_alignment(data, label, return_col):
    ret = pd.to_numeric(data[return_col], errors="coerce")

    if label == "bullish":
        signed = ret
    elif label == "bearish":
        signed = -ret
    else:
        signed = -ret.abs()

    signed = signed.dropna()

    if len(signed) < 3:
        return np.nan, np.nan

    if label == "neutral":
        p = np.nan
    else:
        p = binomtest(
            int((signed > 0).sum()),
            len(signed),
            0.5,
            alternative="greater",
        ).pvalue

    return float(100 * signed.mean()), float(p) if np.isfinite(p) else np.nan


def empirical_rule_stats(data, condition_cols, label, return_col):
    if len(data) == 0:
        return {
            "n": 0,
            "accuracy_pct": np.nan,
            "aligned_mean_pct": np.nan,
            "aligned_signflip_p": np.nan,
        }

    active = np.ones(len(data), dtype=bool)

    for col in condition_cols:
        active &= (
            pd.to_numeric(data[col], errors="coerce")
            .fillna(0)
            .to_numpy()
            >= ANTECEDENT_THRESHOLD
        )

    subset = data.loc[active].copy()

    if subset.empty:
        return {
            "n": 0,
            "accuracy_pct": np.nan,
            "aligned_mean_pct": np.nan,
            "aligned_signflip_p": np.nan,
        }

    aligned_mean, p = label_return_alignment(subset, label, return_col)

    return {
        "n": int(len(subset)),
        "accuracy_pct": float(100 * subset["target_label"].eq(label).mean()),
        "aligned_mean_pct": aligned_mean,
        "aligned_signflip_p": p,
    }


def infer_rule_block(feature_name):
    if feature_name.startswith("interaction__"):
        return "qwen_context_interaction"

    context_feature_names = {
        "ticker_prior_up_5d", "ticker_prior_down_5d",
        "market_prior_up_5d", "market_prior_down_5d",
        "relative_prior_up_5d", "relative_prior_down_5d",
        "quiet_market_5d", "volatile_market_5d",
        "quiet_ticker_5d", "volatile_ticker_5d",
        "event_density_ticker_7d_01", "event_density_ticker_30d_01",
        "similar_density_ticker_30d_01", "novel_context_30d",
        "hour_sin_01", "hour_cos_01", "is_monday", "is_friday",
    }

    if feature_name in context_feature_names:
        return "market_context"

    qwen_prefixes = (
        "event_type_",
        "direction_",
        "confidence_",
        "company_relevance_",
        "log1p_",
        "article_event_weight_",
    )

    if feature_name.startswith(qwen_prefixes) or feature_name == "has_article_event":
        return "qwen_semantic"

    return "unknown"

def mine_validated_rules(
    data,
    feature_cols,
    return_col,
    rule_blocks=None,
    max_pair_features=36,
    max_rules_per_label=12,
):
    training_df = data.loc[data["period"].eq("training")].copy()
    validation_df = data.loc[data["period"].eq("validation")].copy()

    cols = [c for c in feature_cols if c in data.columns]

    if rule_blocks is not None:
        allowed_blocks = set(rule_blocks)
        cols = [c for c in cols if infer_rule_block(c) in allowed_blocks]

    active_counts = (
        training_df[cols]
        .ge(ANTECEDENT_THRESHOLD)
        .sum()
        .sort_values(ascending=False)
    )

    pair_base = active_counts.head(max_pair_features).index.tolist()
    candidates = [(c,) for c in cols] + list(combinations(pair_base, 2))

    rows = []

    for condition_cols in candidates:
        condition_cols = tuple(condition_cols)

        for label in LABELS:
            disc = empirical_rule_stats(
                training_df,
                condition_cols,
                label,
                return_col,
            )

            if disc["n"] < MIN_TRAINING_EVENTS or not np.isfinite(disc["accuracy_pct"]):
                continue

            val = empirical_rule_stats(
                validation_df,
                condition_cols,
                label,
                return_col,
            )

            if val["n"] < MIN_VALIDATION_EVENTS or not np.isfinite(val["accuracy_pct"]):
                continue

            score = (
                disc["accuracy_pct"] / 100
                + val["accuracy_pct"] / 100
                + min(disc["n"] / 40, 1)
                + min(val["n"] / 30, 1)
            )

            if label != "neutral" and np.isfinite(disc["aligned_mean_pct"]):
                score += min(max(disc["aligned_mean_pct"], 0) / 2, 1)

            if label == "neutral" and np.isfinite(disc["aligned_mean_pct"]):
                score += min(max(-disc["aligned_mean_pct"], 0) / 1, 1)

            rows.append({
                "condition": " & ".join(condition_cols),
                "condition_cols_json": json.dumps(condition_cols),
                "rule_label": label,
                "rule_score": score,
                "rule_blocks": ", ".join(
                    sorted({infer_rule_block(c) for c in condition_cols})
                ),
                **{f"training_{k}": v for k, v in disc.items()},
                **{f"validation_{k}": v for k, v in val.items()},
            })

    candidates_df = pd.DataFrame(rows)

    if candidates_df.empty:
        return candidates_df, candidates_df

    selected = candidates_df.loc[
        candidates_df["training_accuracy_pct"].ge(MIN_TRAINING_ACCURACY)
        & candidates_df["validation_accuracy_pct"].ge(MIN_VALIDATION_ACCURACY)
    ].copy()

    selected = selected.sort_values(
        ["rule_label", "validation_accuracy_pct", "validation_n", "rule_score"],
        ascending=[True, False, False, False],
    )

    selected = selected.groupby("rule_label", group_keys=False).head(max_rules_per_label)

    selected = selected.sort_values(
        ["validation_accuracy_pct", "validation_n", "rule_score"],
        ascending=False,
    ).reset_index(drop=True)

    return (
        candidates_df.sort_values("rule_score", ascending=False).reset_index(drop=True),
        selected,
    )

def mine_training_only_rules(
    data,
    feature_cols,
    return_col,
    rule_blocks=None,
    max_pair_features=36,
    max_rules_per_label=12,
):
    training_df = data.loc[data["period"].eq("training")].copy()

    cols = [c for c in feature_cols if c in data.columns]

    if rule_blocks is not None:
        allowed_blocks = set(rule_blocks)
        cols = [c for c in cols if infer_rule_block(c) in allowed_blocks]

    active_counts = (
        training_df[cols]
        .ge(ANTECEDENT_THRESHOLD)
        .sum()
        .sort_values(ascending=False)
    )

    pair_base = active_counts.head(max_pair_features).index.tolist()
    candidates = [(c,) for c in cols] + list(combinations(pair_base, 2))

    rows = []

    for condition_cols in candidates:
        condition_cols = tuple(condition_cols)

        for label in LABELS:
            disc = empirical_rule_stats(
                training_df,
                condition_cols,
                label,
                return_col,
            )

            if disc["n"] < MIN_TRAINING_EVENTS or not np.isfinite(disc["accuracy_pct"]):
                continue

            score = disc["accuracy_pct"] / 100 + min(disc["n"] / 40, 1)

            rows.append({
                "condition": " & ".join(condition_cols),
                "condition_cols_json": json.dumps(condition_cols),
                "rule_label": label,
                "rule_score": score,
                "rule_selection_scope": "training_only",
                "rule_blocks": ", ".join(sorted({infer_rule_block(c) for c in condition_cols})),
                **{f"training_{k}": v for k, v in disc.items()},
            })

    candidates_df = pd.DataFrame(rows)

    if candidates_df.empty:
        return candidates_df, candidates_df

    selected = candidates_df.loc[
        candidates_df["training_accuracy_pct"].ge(MIN_TRAINING_ACCURACY)
    ].copy()

    selected = selected.sort_values(
        ["rule_label", "training_accuracy_pct", "training_n", "rule_score"],
        ascending=[True, False, False, False],
    )

    selected = selected.groupby("rule_label", group_keys=False).head(max_rules_per_label)

    selected = selected.sort_values(
        ["training_accuracy_pct", "training_n", "rule_score"],
        ascending=False,
    ).reset_index(drop=True)

    return (
        candidates_df.sort_values("rule_score", ascending=False).reset_index(drop=True),
        selected,
    )


class FeaturePredicate(nn.Module):
    def __init__(self, index):
        super().__init__()
        self.index = index

    def forward(self, z):
        return z[:, self.index].reshape(-1, 1)


def make_feature_predicates(feature_cols):
    preds = {}

    for i, col in enumerate(feature_cols):
        preds[col] = ltn.Predicate(model=FeaturePredicate(i).to(device))

    return preds


def antecedent_from_rule(cols, feat_preds, x, AndOp):
    expr = truth_value(feat_preds[cols[0]](x))

    for col in cols[1:]:
        expr = AndOp(expr, truth_value(feat_preds[col](x)))

    return expr


def train_variant(
    name,
    spec,
    train_df,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=True,
):
    reset_seed(seed)

    feature_cols = spec["feature_cols"]
    return_col = spec["return_col"]

    if use_logic and mine_rules:
        candidates, selected_rules = mine_validated_rules(
            train_df,
            feature_cols,
            return_col,
            rule_blocks=spec.get("rule_blocks"),
            max_pair_features=spec.get("max_pair_features", 36),
            max_rules_per_label=spec.get("max_rules_per_label", 12),
        )

    elif use_logic and not mine_rules:
        candidates = pd.DataFrame()

        if "selected_rules" not in spec:
            raise ValueError(
                f"{name} was called with mine_rules=False, "
                "but spec['selected_rules'] was not provided."
            )

        selected_rules = spec["selected_rules"].copy()

    else:
        candidates = pd.DataFrame()
        selected_rules = pd.DataFrame()

    if SAVE_OUTPUTS and use_logic:
        candidates.to_csv(
            OUTPUT_DIR / f"{name}_candidate_mined_rules.csv",
            index=False,
        )
        selected_rules.to_csv(
            OUTPUT_DIR / f"{name}_selected_validated_rules.csv",
            index=False,
        )

    X = feature_tensor(train_df, feature_cols)
    x = ltn.Variable(f"x_{name}", X)
    y = target_tensor(train_df)

    feat_preds = make_feature_predicates(feature_cols)

    classifier = OutcomeSoftmaxMLP(X.shape[1], n_classes=len(LABELS)).to(device)

    Bearish = ltn.Predicate(model=ClassPredicate(classifier, 0).to(device))
    Bullish = ltn.Predicate(model=ClassPredicate(classifier, 1).to(device))
    Neutral = ltn.Predicate(model=ClassPredicate(classifier, 2).to(device))

    AndOp = ltn.fuzzy_ops.AndProd()
    ImpliesOp = ltn.fuzzy_ops.ImpliesReichenbach()
    ForallAgg = ltn.fuzzy_ops.AggregPMeanError(p=2)
    SatAgg = ltn.fuzzy_ops.SatAgg()

    label_predicates = {
        "bearish": Bearish,
        "bullish": Bullish,
        "neutral": Neutral,
    }

    rule_specs = []

    for _, rule in selected_rules.iterrows():
        cols = [
            c for c in json.loads(rule["condition_cols_json"])
            if c in feat_preds
        ]

        if cols:
            rule_specs.append((rule["condition"], cols, rule["rule_label"]))

    opt = torch.optim.Adam(classifier.parameters(), lr=LR)

    history = []

    for epoch in range(EPOCHS):
        opt.zero_grad()

        logits = classifier(X)

        pred_bear = truth_value(Bearish(x)).reshape(-1, 1)
        pred_bull = truth_value(Bullish(x)).reshape(-1, 1)
        pred_neut = truth_value(Neutral(x)).reshape(-1, 1)

        supervised = F.cross_entropy(logits, y)

        exclusivity = torch.tensor(0.0, device=device)

        formulas = []

        if use_logic:
            for rule_name, cols, label in rule_specs:
                formulas.append((
                    rule_name,
                    ForallAgg(
                        ImpliesOp(
                            antecedent_from_rule(cols, feat_preds, x, AndOp),
                            truth_value(label_predicates[label](x)),
                        )
                    ),
                ))

        sat = (
            SatAgg(*[formula for _, formula in formulas])
            if formulas
            else torch.tensor(1.0, device=device)
        )

        logic_penalty = (
            logic_weight * (1.0 - truth_value(sat))
            if use_logic
            else torch.tensor(0.0, device=device)
        )

        loss = supervised + logic_penalty
        loss.backward()
        opt.step()

        if epoch % 50 == 0 or epoch == EPOCHS - 1:
            row = {
                "variant": name,
                "epoch": epoch,
                "seed": seed,
                "use_logic": use_logic,
                "mine_rules": mine_rules,
                "logic_weight": logic_weight,
                "loss": float(loss.detach().cpu()),
                "supervised_cross_entropy": float(supervised.detach().cpu()),
                "exclusivity": float(exclusivity.detach().cpu()),
                "sat": float(truth_value(sat).detach().cpu())
                if hasattr(truth_value(sat), "detach")
                else float(sat),
                "n_selected_rules": len(rule_specs),
            }

            for rule_name, formula in formulas[:20]:
                row[f"sat_{rule_name[:80]}"] = float(
                    truth_value(formula).detach().cpu()
                )

            history.append(row)

    return {
        "name": name,
        "seed": seed,
        "feature_cols": feature_cols,
        "candidate_rules": candidates,
        "selected_rules": selected_rules,
        "use_logic": use_logic,
        "mine_rules": mine_rules,
        "logic_weight": logic_weight,
        "classifier": classifier,
        "Bearish": Bearish,
        "Bullish": Bullish,
        "Neutral": Neutral,
        "history": pd.DataFrame(history),
    }

In [7]:
# --- Fix: cluster_bootstrap_aligned_stats collided with an existing
# "prediction" column when true_selective_subset_stats renamed
# selective_prediction -> prediction (score_variant already added its own
# "prediction" column, so the rename created two columns with the same
# name, and df["prediction"] returned a DataFrame instead of a Series).
# Fix: take the prediction column name as a parameter instead of renaming.

def cluster_bootstrap_accuracy_ci(scored_df, cluster_cols=("ticker", "event_date"), prediction_col="prediction", n_boot=5000, seed=0):
    df = scored_df.copy()
    df["_correct"] = (df[prediction_col] == df["target_label"]).astype(int)

    per_cluster = df.groupby(list(cluster_cols))["_correct"].agg(["sum", "count"]).reset_index()
    correct = per_cluster["sum"].to_numpy()
    total = per_cluster["count"].to_numpy()
    n_clusters = len(per_cluster)

    rng = np.random.default_rng(seed)
    boot_acc = np.empty(n_boot)

    for b in range(n_boot):
        idx = rng.integers(0, n_clusters, size=n_clusters)
        boot_acc[b] = correct[idx].sum() / total[idx].sum()

    return {
        "n_independent_clusters": n_clusters,
        "accuracy_ci_low_pct": float(100 * np.percentile(boot_acc, 2.5)),
        "accuracy_ci_high_pct": float(100 * np.percentile(boot_acc, 97.5)),
    }


def cluster_bootstrap_aligned_stats(scored_df, cluster_cols=("ticker", "event_date"), prediction_col="prediction", n_boot=5000, seed=0):
    df = scored_df.copy()
    ret = pd.to_numeric(df["simple_return"], errors="coerce")
    pred = df[prediction_col]

    df["_signed"] = np.select(
        [pred.eq("bullish"), pred.eq("bearish")],
        [ret, -ret],
        default=np.nan,
    )
    df = df.dropna(subset=["_signed"])

    if df.empty:
        return {
            "n_independent_clusters": 0,
            "aligned_mean_ci_low_pct": np.nan,
            "aligned_mean_ci_high_pct": np.nan,
            "aligned_bootstrap_p_value": np.nan,
        }

    per_cluster = df.groupby(list(cluster_cols))["_signed"].agg(["sum", "count"]).reset_index()
    cluster_sum = per_cluster["sum"].to_numpy()
    cluster_count = per_cluster["count"].to_numpy()
    n_clusters = len(per_cluster)

    rng = np.random.default_rng(seed)
    boot_means = np.empty(n_boot)

    for b in range(n_boot):
        idx = rng.integers(0, n_clusters, size=n_clusters)
        boot_means[b] = cluster_sum[idx].sum() / cluster_count[idx].sum()

    return {
        "n_independent_clusters": n_clusters,
        "aligned_mean_ci_low_pct": float(100 * np.percentile(boot_means, 2.5)),
        "aligned_mean_ci_high_pct": float(100 * np.percentile(boot_means, 97.5)),
        "aligned_bootstrap_p_value": float((boot_means <= 0).mean()),
    }

In [8]:
def score_variant(bundle, df):
    out = df.copy()
    X = feature_tensor(out, bundle["feature_cols"])

    with torch.no_grad():
        logits = bundle["classifier"](X)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()

    out["score_bearish"] = probs[:, 0]
    out["score_bullish"] = probs[:, 1]
    out["score_neutral"] = probs[:, 2]

    scores = out[["score_bearish", "score_bullish", "score_neutral"]].to_numpy()

    out["prediction"] = np.array(LABELS)[scores.argmax(axis=1)]

    out["confidence_margin"] = (
        np.sort(scores, axis=1)[:, -1]
        - np.sort(scores, axis=1)[:, -2]
    )

    out["max_class_score"] = scores.max(axis=1)

    return out


def aligned_stats(df):
    ret = pd.to_numeric(df["simple_return"], errors="coerce")
    pred = df["prediction"]

    signed = np.select(
        [pred.eq("bullish"), pred.eq("bearish")],
        [ret, -ret],
        default=np.nan,
    )

    signed = pd.Series(signed, index=df.index).dropna()

    if len(signed) < 3:
        return np.nan, np.nan

    p = binomtest(
        int((signed > 0).sum()),
        len(signed),
        0.5,
        alternative="greater",
    ).pvalue

    return float(signed.mean() * 100), float(p)


def summary_row(variant, period, df, bundle=None):
    aligned_mean, aligned_p = aligned_stats(df)

    # aligned_signflip_p below is the original article-level binomial test,
    # kept for continuity. It is not valid when df contains multiple articles
    # per (ticker, event_date) cluster (see Section 4). The cluster-bootstrap
    # fields (n_independent_clusters, aligned_mean_ci_*, aligned_bootstrap_p_value)
    # are the corrected statistics and are what should be reported/cited.
    cluster_stats = cluster_bootstrap_aligned_stats(df)

    row = {
        "variant": variant,
        "period": period,
        "n": int(len(df)),
        "accuracy_pct": float(100 * df["prediction"].eq(df["target_label"]).mean()),
        "mean_simple_return_pct": float(
            100 * pd.to_numeric(df["simple_return"], errors="coerce").mean()
        ),
        "aligned_mean_pct": aligned_mean,
        "aligned_signflip_p": aligned_p,
        "bullish_pred_rate_pct": float(100 * df["prediction"].eq("bullish").mean()),
        "bearish_pred_rate_pct": float(100 * df["prediction"].eq("bearish").mean()),
        "neutral_pred_rate_pct": float(100 * df["prediction"].eq("neutral").mean()),
        "mean_confidence_margin": float(df["confidence_margin"].mean()),
        "mean_max_class_score": float(df["max_class_score"].mean()),
        **cluster_stats,
    }

    if bundle is not None:
        row["use_logic"] = bundle["use_logic"]
        row["logic_weight"] = bundle["logic_weight"]
        row["n_selected_rules"] = len(bundle["selected_rules"])

    return row


## 7. Validation-Only Horizon Audit

This section compares alternative MAG7 return horizons using training and validation data only. The held-out test set is not used for horizon selection.

In [9]:

HORIZON_AUDIT_VARIANT = PRIMARY_VARIANT


def build_model_df_for_horizon(horizon: int) -> pd.DataFrame:
    horizon_returns = simple_returns.loc[
        simple_returns["horizon_days"].eq(int(horizon)),
        ["ticker", "event_date", "simple_return", "event_open", "event_close"],
    ]

    out = (
        feature_df
        .drop(
            columns=[
                "simple_return",
                "prev_close",
                "future_close",
                "event_open",
                "event_close",
                "target_label",
            ],
            errors="ignore",
        )
        .merge(
            horizon_returns,
            on=["ticker", "event_date"],
            how="left",
        )
    ).copy()

    missing_returns = int(out["simple_return"].isna().sum())
    if missing_returns:
        print(f"{horizon}d horizon: dropping {missing_returns} rows with missing open-to-close returns")
        out = out.loc[out["simple_return"].notna()].copy()

    out["target_label"] = out["simple_return"].map(
        lambda x: realised_label_from_return(float(x), RETURN_THRESHOLD)
    )

    return out


horizon_ltn_rows = []

for horizon in HORIZON_CANDIDATES:
    print(f"LTN horizon audit: open to {horizon}-day close return")

    model_df_h = build_model_df_for_horizon(horizon)

    train_h = model_df_h.loc[model_df_h["period"].eq("training")].copy()
    validation_h = model_df_h.loc[model_df_h["period"].eq("validation")].copy()

    spec_base_h = variant_specs[HORIZON_AUDIT_VARIANT].copy()

    candidate_rules_h, selected_rules_h = mine_training_only_rules(
        train_h,
        spec_base_h["feature_cols"],
        spec_base_h["return_col"],
        rule_blocks=spec_base_h.get("rule_blocks"),
        max_pair_features=spec_base_h.get("max_pair_features", 36),
        max_rules_per_label=spec_base_h.get("max_rules_per_label", 12),
    )

    spec_h = spec_base_h.copy()
    spec_h["selected_rules"] = selected_rules_h.copy()

    bundle_h = train_variant(
        f"{HORIZON_AUDIT_VARIANT}_horizon_{horizon}d",
        spec_h,
        train_h,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=SEED,
        mine_rules=False,
    )

    scored_validation_h = score_variant(bundle_h, validation_h)

    row_h = summary_row(
        f"LTN horizon {horizon}d",
        "validation",
        scored_validation_h,
        bundle=bundle_h,
    )

    majority_baseline_h = validation_h["target_label"].value_counts(normalize=True).max() * 100

    row_h["horizon_days"] = horizon
    row_h["training_n"] = len(train_h)
    row_h["validation_n"] = len(validation_h)
    row_h["validation_majority_baseline_pct"] = float(majority_baseline_h)
    row_h["validation_bullish_n"] = int(validation_h["target_label"].eq("bullish").sum())
    row_h["validation_bearish_n"] = int(validation_h["target_label"].eq("bearish").sum())
    row_h["validation_neutral_n"] = int(validation_h["target_label"].eq("neutral").sum())
    row_h["n_candidate_rules"] = len(candidate_rules_h)
    row_h["n_selected_rules"] = len(selected_rules_h)

    horizon_ltn_rows.append(row_h)


horizon_ltn_audit = pd.DataFrame(horizon_ltn_rows)

display(
    horizon_ltn_audit.sort_values(
        ["accuracy_pct", "aligned_mean_pct", "n_selected_rules"],
        ascending=False,
    )
)


LTN horizon audit: open to 1-day close return
LTN horizon audit: open to 2-day close return
2d horizon: dropping 57 rows with missing open-to-close returns
LTN horizon audit: open to 3-day close return
3d horizon: dropping 99 rows with missing open-to-close returns
LTN horizon audit: open to 4-day close return
4d horizon: dropping 147 rows with missing open-to-close returns
LTN horizon audit: open to 5-day close return
5d horizon: dropping 182 rows with missing open-to-close returns


,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,mean_max_class_score,n_independent_clusters,aligned_mean_ci_low_pct,aligned_mean_ci_high_pct,aligned_bootstrap_p_value,use_logic,logic_weight,n_selected_rules,horizon_days,training_n,validation_n,validation_majority_baseline_pct,validation_bullish_n,validation_bearish_n,validation_neutral_n,n_candidate_rules
4,LTN horizon 5d,validation,1966,49.491353,-1.141053,0.384662,0.000001,51.983723,41.505595,6.510682,0.804992,0.896303,251,-0.001545,0.781834,0.0260,True,0.3,24,5,7349,1966,54.984741,801,1081,84,2139
3,LTN horizon 4d,validation,1966,47.049847,-0.963437,0.280477,0.000309,50.101729,38.708037,11.190234,0.820375,0.905339,256,-0.154781,0.718979,0.0952,True,0.3,26,4,7349,1966,53.611394,796,1054,116,2139
1,LTN horizon 2d,validation,1966,43.540183,-0.302738,0.184952,0.070216,33.519837,56.459817,10.020346,0.836854,0.913517,253,-0.095117,0.452145,0.0900,True,0.3,36,2,7349,1966,47.456765,741,933,292,2139
2,LTN horizon 3d,validation,1966,35.757884,-0.652869,-0.062853,0.980958,47.660224,35.808749,16.531027,0.808816,0.898492,255,-0.373492,0.241830,0.6648,True,0.3,36,3,7349,1966,48.728383,744,958,264,2139
0,LTN horizon 1d,validation,1966,35.198372,-0.202131,0.009706,0.157322,31.230926,37.639878,31.129196,0.832359,0.911482,248,-0.181880,0.213152,0.4580,True,0.3,36,1,7349,1966,43.133266,611,848,507,2139


**Reading this table:** 5-day return wins on both accuracy (49.5%) and aligned mean return (0.385%), and is the only horizon with a cluster-bootstrap p-value below 0.05 (p = 0.026). 4-day is a distant second (47.0% accuracy, p = 0.095, not significant). This is why `PRIMARY_HORIZON = 5` in Section 8.

## 8. Build Final Labelled Modelling Frame

This section attaches the retained five-day return label to the feature matrix and creates the final training, validation and test datasets.

In [10]:
# The validation-only horizon audit in Section 7 ranks the 5-day
# open-to-close horizon highest on both accuracy (49.5%) and aligned
# mean return (0.385%), and is the only horizon that clears
# conventional significance under the cluster-bootstrap test
# (p = 0.026). It is used here as the primary horizon.
PRIMARY_HORIZON = 5


In [11]:

model_df = build_model_df_for_horizon(PRIMARY_HORIZON)

train = model_df.loc[model_df["period"].eq("training")].copy()
validation = model_df.loc[model_df["period"].eq("validation")].copy()
test = model_df.loc[model_df["period"].eq("test")].copy()

display(model_df.groupby("period")["target_label"].value_counts().unstack(fill_value=0))

5d horizon: dropping 182 rows with missing open-to-close returns


target_label,bearish,bullish,neutral
period,,,
test,1108,1412,255
training,2847,3936,566
validation,1081,801,84


## 9. Full Classifier: Frozen-Rule Test Evaluation

Rules are selected using training and validation data, frozen, and then used in the final refit before held-out test evaluation.

In [12]:
train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()

frozen_variant_specs = {}

for name, spec in variant_specs.items():
    print("selecting frozen rules", name)

    candidate_rules, selected_rules = mine_validated_rules(
        train_val,
        spec["feature_cols"],
        spec["return_col"],
        rule_blocks=spec.get("rule_blocks"),
        max_pair_features=spec.get("max_pair_features", 36),
        max_rules_per_label=spec.get("max_rules_per_label", 12),
    )

    frozen_spec = spec.copy()
    frozen_spec["selected_rules"] = selected_rules.copy()
    frozen_variant_specs[name] = frozen_spec


trained_final = {}

for name, spec in frozen_variant_specs.items():
    print("final frozen-rule training", name, "rows", len(train_val))

    trained_final[name] = train_variant(
        name,
        spec,
        train_val,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=SEED,
        mine_rules=False,
    )


final_rows = []
final_frames = []

for variant, bundle in trained_final.items():
    scored = score_variant(bundle, test)

    final_rows.append(
        summary_row(
            variant,
            "test_frozen_rules_refit_training_validation",
            scored,
            bundle=bundle,
        )
    )

    final_frames.append(
        scored.assign(
            variant=variant,
            period="test_frozen_rules_refit_training_validation",
        )
    )


final_summary = pd.DataFrame(final_rows)

if SAVE_OUTPUTS:
    final_summary.to_csv(
        OUTPUT_DIR / "full_ltn_final_frozen_rule_test_summary.csv",
        index=False,
    )

    pd.concat(final_frames, ignore_index=True).to_csv(
        OUTPUT_DIR / "full_ltn_final_frozen_rule_test_predictions.csv",
        index=False,
    )


display(
    final_summary.sort_values(
        ["accuracy_pct", "aligned_mean_pct"],
        ascending=False,
    )
)

selecting frozen rules full_ltn_qwen_only
selecting frozen rules full_ltn_context_only
selecting frozen rules full_ltn_qwen_plus_context
selecting frozen rules full_ltn_qwen_context_interaction
final frozen-rule training full_ltn_qwen_only rows 9315
final frozen-rule training full_ltn_context_only rows 9315
final frozen-rule training full_ltn_qwen_plus_context rows 9315
final frozen-rule training full_ltn_qwen_context_interaction rows 9315


,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,mean_max_class_score,n_independent_clusters,aligned_mean_ci_low_pct,aligned_mean_ci_high_pct,aligned_bootstrap_p_value,use_logic,logic_weight,n_selected_rules
1,full_ltn_context_only,test_frozen_rules_refit_training_validation,2775,47.675676,1.09779,0.652623,3.400240e-08,50.810811,41.585586,7.603604,0.599941,0.776258,343,0.051302,1.266928,0.0168,True,0.3,14
2,full_ltn_qwen_plus_context,test_frozen_rules_refit_training_validation,2775,44.432432,1.09779,0.296396,1.742616e-02,53.009009,39.675676,7.315315,0.744739,0.863419,347,-0.292241,0.906229,0.1606,True,0.3,24
0,full_ltn_qwen_only,test_frozen_rules_refit_training_validation,2775,42.882883,1.09779,-0.207448,9.908041e-01,42.306306,57.405405,0.288288,0.227886,0.577994,349,-0.525429,0.096508,0.9046,True,0.3,7
3,full_ltn_qwen_context_interaction,test_frozen_rules_refit_training_validation,2775,42.594595,1.09779,0.230722,6.828650e-01,57.441441,34.270270,8.288288,0.796798,0.890834,345,-0.325220,0.801852,0.2124,True,0.3,24


**Reading this table:** `full_ltn_context_only` -- the variant built purely from price and market-context features, with no LLM-derived event features at all -- is both the most accurate variant (47.7%) and the only one whose aligned return clears significance under the cluster-bootstrap test (p = 0.017). `full_ltn_qwen_context_interaction`, the variant used throughout the ablations and selective-classifier sections below as `PRIMARY_VARIANT`, is the *least* accurate of the four (42.6%) and is not significant (p = 0.21) on this test window. Section 16 revisits this table alongside the Section 13 robustness-window results, where the ranking reverses.

In [13]:

RETURN_THRESHOLD_CANDIDATES = [0.005, 0.0075, 0.01, 0.0125, 0.015, 0.02]
THRESHOLD_AUDIT_HORIZON = PRIMARY_HORIZON
THRESHOLD_AUDIT_VARIANT = PRIMARY_VARIANT
THRESHOLD_AUDIT_SEED = SEED
THRESHOLD_AUDIT_EPOCHS = 250  # speed-up for audit; final model can use full EPOCHS

def realised_label_from_return_at_threshold(value, threshold):
    if pd.isna(value):
        return "neutral"
    if value > threshold:
        return "bullish"
    if value < -threshold:
        return "bearish"
    return "neutral"


def build_model_df_for_horizon_and_threshold(horizon, threshold):
    horizon_returns = simple_returns.loc[
        simple_returns["horizon_days"].eq(int(horizon)),
        ["ticker", "event_date", "simple_return", "event_open", "event_close"],
    ]

    out = (
        feature_df
        .drop(
            columns=[
                "simple_return",
                "prev_close",
                "future_close",
                "event_open",
                "event_close",
                "target_label",
            ],
            errors="ignore",
        )
        .merge(
            horizon_returns,
            on=["ticker", "event_date"],
            how="left",
        )
    ).copy()

    missing_returns = int(out["simple_return"].isna().sum())

    if missing_returns:
        print(
            f"threshold audit: horizon {horizon}, threshold {threshold}: "
            f"dropping {missing_returns} rows with missing open-to-close returns"
        )
        out = out.loc[out["simple_return"].notna()].copy()

    out["target_label"] = out["simple_return"].map(
        lambda x: realised_label_from_return_at_threshold(float(x), threshold)
    )

    return out


threshold_audit_rows = []
threshold_audit_label_rows = []

old_epochs = EPOCHS
EPOCHS = THRESHOLD_AUDIT_EPOCHS

try:
    for threshold in RETURN_THRESHOLD_CANDIDATES:
        print(f"Return-threshold audit: threshold = {threshold:.4f}")

        model_df_t = build_model_df_for_horizon_and_threshold(
            THRESHOLD_AUDIT_HORIZON,
            threshold,
        )

        train_t = model_df_t.loc[model_df_t["period"].eq("training")].copy()
        validation_t = model_df_t.loc[model_df_t["period"].eq("validation")].copy()

        label_counts = (
            model_df_t
            .groupby("period")["target_label"]
            .value_counts()
            .unstack(fill_value=0)
            .reindex(columns=LABELS, fill_value=0)
            .reset_index()
        )

        label_counts["return_threshold"] = threshold
        label_counts["horizon_days"] = THRESHOLD_AUDIT_HORIZON
        threshold_audit_label_rows.append(label_counts)

        majority_baseline = (
            validation_t["target_label"]
            .value_counts(normalize=True)
            .max()
            * 100
        )

        spec_base = variant_specs[THRESHOLD_AUDIT_VARIANT].copy()

        candidate_rules_t, selected_rules_t = mine_training_only_rules(
            train_t,
            spec_base["feature_cols"],
            spec_base["return_col"],
            rule_blocks=spec_base.get("rule_blocks"),
            max_pair_features=spec_base.get("max_pair_features", 36),
            max_rules_per_label=spec_base.get("max_rules_per_label", 12),
        )

        spec_t = spec_base.copy()
        spec_t["selected_rules"] = selected_rules_t.copy()

        ltn_bundle_t = train_variant(
            f"{THRESHOLD_AUDIT_VARIANT}_return_threshold_{str(threshold).replace('.', 'p')}",
            spec_t,
            train_t,
            use_logic=True,
            logic_weight=LOGIC_WEIGHT,
            seed=THRESHOLD_AUDIT_SEED,
            mine_rules=False,
        )

        ltn_validation_scored = score_variant(ltn_bundle_t, validation_t)
        ltn_row = summary_row(
            f"LTN threshold {threshold:.4f}",
            "validation",
            ltn_validation_scored,
            bundle=ltn_bundle_t,
        )

        ltn_row.update({
            "return_threshold": threshold,
            "horizon_days": THRESHOLD_AUDIT_HORIZON,
            "model": "LTN, article-event + context interaction",
            "validation_majority_baseline_pct": float(majority_baseline),
            "training_n": len(train_t),
            "validation_n": len(validation_t),
            "training_bearish_n": int(train_t["target_label"].eq("bearish").sum()),
            "training_bullish_n": int(train_t["target_label"].eq("bullish").sum()),
            "training_neutral_n": int(train_t["target_label"].eq("neutral").sum()),
            "validation_bearish_n": int(validation_t["target_label"].eq("bearish").sum()),
            "validation_bullish_n": int(validation_t["target_label"].eq("bullish").sum()),
            "validation_neutral_n": int(validation_t["target_label"].eq("neutral").sum()),
            "n_candidate_rules": len(candidate_rules_t),
            "n_selected_rules": len(selected_rules_t),
            "accuracy_minus_majority_pct": float(
                ltn_row["accuracy_pct"] - majority_baseline
            ),
            "selection_score": float(
                ltn_row["accuracy_pct"]
                - majority_baseline
                + max(ltn_row["aligned_mean_pct"], 0)
            ),
        })

        threshold_audit_rows.append(ltn_row)

        no_logic_bundle_t = train_variant(
            f"no_logic_return_threshold_{str(threshold).replace('.', 'p')}",
            spec_base,
            train_t,
            use_logic=False,
            logic_weight=0.0,
            seed=THRESHOLD_AUDIT_SEED,
            mine_rules=False,
        )

        no_logic_validation_scored = score_variant(no_logic_bundle_t, validation_t)
        no_logic_row = summary_row(
            f"No-logic threshold {threshold:.4f}",
            "validation",
            no_logic_validation_scored,
            bundle=no_logic_bundle_t,
        )

        no_logic_row.update({
            "return_threshold": threshold,
            "horizon_days": THRESHOLD_AUDIT_HORIZON,
            "model": "Neural classifier, Qwen + context interaction, no logic",
            "validation_majority_baseline_pct": float(majority_baseline),
            "training_n": len(train_t),
            "validation_n": len(validation_t),
            "training_bearish_n": int(train_t["target_label"].eq("bearish").sum()),
            "training_bullish_n": int(train_t["target_label"].eq("bullish").sum()),
            "training_neutral_n": int(train_t["target_label"].eq("neutral").sum()),
            "validation_bearish_n": int(validation_t["target_label"].eq("bearish").sum()),
            "validation_bullish_n": int(validation_t["target_label"].eq("bullish").sum()),
            "validation_neutral_n": int(validation_t["target_label"].eq("neutral").sum()),
            "n_candidate_rules": 0,
            "n_selected_rules": 0,
            "accuracy_minus_majority_pct": float(
                no_logic_row["accuracy_pct"] - majority_baseline
            ),
            "selection_score": float(
                no_logic_row["accuracy_pct"]
                - majority_baseline
                + max(no_logic_row["aligned_mean_pct"], 0)
            ),
        })

        threshold_audit_rows.append(no_logic_row)

finally:
    EPOCHS = old_epochs


return_threshold_audit = pd.DataFrame(threshold_audit_rows)
return_threshold_label_counts = pd.concat(
    threshold_audit_label_rows,
    ignore_index=True,
)

display(
    return_threshold_label_counts[
        [
            "return_threshold",
            "horizon_days",
            "period",
            "bearish",
            "bullish",
            "neutral",
        ]
    ]
)

display(
    return_threshold_audit[
        [
            "model",
            "return_threshold",
            "horizon_days",
            "validation_n",
            "accuracy_pct",
            "validation_majority_baseline_pct",
            "accuracy_minus_majority_pct",
            "aligned_mean_pct",
            "aligned_signflip_p",
            "bullish_pred_rate_pct",
            "bearish_pred_rate_pct",
            "neutral_pred_rate_pct",
            "n_selected_rules",
            "selection_score",
        ]
    ]
    .sort_values(
        ["selection_score", "accuracy_minus_majority_pct", "aligned_mean_pct"],
        ascending=False,
    )
    .reset_index(drop=True)
)

best_return_threshold_config = (
    return_threshold_audit
    .sort_values(
        ["selection_score", "accuracy_minus_majority_pct", "aligned_mean_pct"],
        ascending=False,
    )
    .iloc[0]
    .to_dict()
)

print("Best validation-selected return-threshold config:")
for key in [
    "model",
    "return_threshold",
    "horizon_days",
    "accuracy_pct",
    "validation_majority_baseline_pct",
    "accuracy_minus_majority_pct",
    "aligned_mean_pct",
    "aligned_signflip_p",
    "n_selected_rules",
    "selection_score",
]:
    print(f"{key}: {best_return_threshold_config.get(key)}")


Return-threshold audit: threshold = 0.0050
threshold audit: horizon 5, threshold 0.005: dropping 182 rows with missing open-to-close returns
Return-threshold audit: threshold = 0.0075
threshold audit: horizon 5, threshold 0.0075: dropping 182 rows with missing open-to-close returns
Return-threshold audit: threshold = 0.0100
threshold audit: horizon 5, threshold 0.01: dropping 182 rows with missing open-to-close returns
Return-threshold audit: threshold = 0.0125
threshold audit: horizon 5, threshold 0.0125: dropping 182 rows with missing open-to-close returns
Return-threshold audit: threshold = 0.0150
threshold audit: horizon 5, threshold 0.015: dropping 182 rows with missing open-to-close returns
Return-threshold audit: threshold = 0.0200
threshold audit: horizon 5, threshold 0.02: dropping 182 rows with missing open-to-close returns


target_label,return_threshold,horizon_days,period,bearish,bullish,neutral
0,0.0050,5,test,1108,1412,255
1,0.0050,5,training,2847,3936,566
2,0.0050,5,validation,1081,801,84
3,0.0075,5,test,1069,1327,379
4,0.0075,5,training,2745,3738,866
5,0.0075,5,validation,1037,774,155
6,0.0100,5,test,1036,1264,475
7,0.0100,5,training,2639,3343,1367
8,0.0100,5,validation,983,696,287
9,0.0125,5,test,988,1191,596


,model,return_threshold,horizon_days,validation_n,accuracy_pct,validation_majority_baseline_pct,accuracy_minus_majority_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,n_selected_rules,selection_score
0,"LTN, article-event + context interaction",0.0050,5,1966,50.050865,54.984741,-4.933876,0.444965,4.300160e-08,50.101729,43.184130,6.714140,24,-4.488911
1,"LTN, article-event + context interaction",0.0100,5,1966,44.150560,50.000000,-5.849440,0.552163,1.491911e-11,47.253306,37.029502,15.717192,27,-5.297278
2,"Neural classifier, Qwen + context interaction,...",0.0050,5,1966,48.168871,54.984741,-6.815870,0.356810,2.772808e-04,51.169888,42.370295,6.459817,0,-6.459060
3,"Neural classifier, Qwen + context interaction,...",0.0075,5,1966,45.625636,52.746694,-7.121058,0.286613,2.874937e-08,52.339776,35.350966,12.309257,0,-6.834445
4,"Neural classifier, Qwen + context interaction,...",0.0100,5,1966,42.421160,50.000000,-7.578840,0.672869,9.812537e-08,43.387589,38.911495,17.700916,0,-6.905971
5,"Neural classifier, Qwen + context interaction,...",0.0200,5,1966,34.333672,41.658189,-7.324517,0.035834,5.435202e-02,36.622584,18.921668,44.455748,0,-7.288683
6,"LTN, article-event + context interaction",0.0075,5,1966,44.913530,52.746694,-7.833164,0.291896,4.661652e-05,47.405900,39.064090,13.530010,24,-7.541267
7,"LTN, article-event + context interaction",0.0125,5,1966,38.860631,46.998983,-8.138352,0.377650,2.483112e-07,44.404883,31.536114,24.059003,36,-7.760702
8,"Neural classifier, Qwen + context interaction,...",0.0125,5,1966,38.809766,46.998983,-8.189217,0.350356,3.495632e-06,47.202442,30.976602,21.820956,0,-7.838860
9,"LTN, article-event + context interaction",0.0200,5,1966,31.790437,41.658189,-9.867752,0.195404,1.317945e-01,34.282808,18.463886,47.253306,30,-9.672348


Best validation-selected return-threshold config:
model: LTN, article-event + context interaction
return_threshold: 0.005
horizon_days: 5
accuracy_pct: 50.05086469989827
validation_majority_baseline_pct: 54.984740590030526
accuracy_minus_majority_pct: -4.933875890132256
aligned_mean_pct: 0.4449650483143916
aligned_signflip_p: 4.30016022781086e-08
n_selected_rules: 24
selection_score: -4.488910841817865


In [14]:

TUNE_VARIANT = PRIMARY_VARIANT
TUNE_SEED = SEED
TUNE_EPOCHS = 250  # reduce for tuning speed; final selected config can be refit later with 500

LOGIC_WEIGHT_GRID = [0.00, 0.01, 0.03, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.00]

RULE_FILTER_GRID = [
    {
        "rule_setting": "current_like",
        "min_training_events": 6,
        "min_validation_events": 5,
        "min_training_accuracy": 60.0,
        "min_validation_accuracy": 60.0,
        "max_rules_per_label": 12,
    },
    {
        "rule_setting": "moderate_support",
        "min_training_events": 10,
        "min_validation_events": 8,
        "min_training_accuracy": 60.0,
        "min_validation_accuracy": 65.0,
        "max_rules_per_label": 10,
    },
    {
        "rule_setting": "strict_accuracy",
        "min_training_events": 10,
        "min_validation_events": 8,
        "min_training_accuracy": 65.0,
        "min_validation_accuracy": 70.0,
        "max_rules_per_label": 8,
    },
    {
        "rule_setting": "high_support_strict",
        "min_training_events": 15,
        "min_validation_events": 10,
        "min_training_accuracy": 65.0,
        "min_validation_accuracy": 70.0,
        "max_rules_per_label": 8,
    },
]

def mine_validated_rules_with_thresholds(
    data,
    feature_cols,
    return_col,
    rule_blocks=None,
    min_training_events=6,
    min_validation_events=5,
    min_training_accuracy=60.0,
    min_validation_accuracy=60.0,
    max_pair_features=36,
    max_rules_per_label=12,
):
    training_df = data.loc[data["period"].eq("training")].copy()
    validation_df = data.loc[data["period"].eq("validation")].copy()

    cols = [c for c in feature_cols if c in data.columns]

    if rule_blocks is not None:
        allowed_blocks = set(rule_blocks)
        cols = [c for c in cols if infer_rule_block(c) in allowed_blocks]

    active_counts = (
        training_df[cols]
        .ge(ANTECEDENT_THRESHOLD)
        .sum()
        .sort_values(ascending=False)
    )

    pair_base = active_counts.head(max_pair_features).index.tolist()
    candidates = [(c,) for c in cols] + list(combinations(pair_base, 2))

    rows = []

    for condition_cols in candidates:
        condition_cols = tuple(condition_cols)

        for label in LABELS:
            train_stats = empirical_rule_stats(
                training_df,
                condition_cols,
                label,
                return_col,
            )

            if (
                train_stats["n"] < min_training_events
                or not np.isfinite(train_stats["accuracy_pct"])
                or train_stats["accuracy_pct"] < min_training_accuracy
            ):
                continue

            val_stats = empirical_rule_stats(
                validation_df,
                condition_cols,
                label,
                return_col,
            )

            if (
                val_stats["n"] < min_validation_events
                or not np.isfinite(val_stats["accuracy_pct"])
                or val_stats["accuracy_pct"] < min_validation_accuracy
            ):
                continue

            score = (
                train_stats["accuracy_pct"] / 100
                + val_stats["accuracy_pct"] / 100
                + min(train_stats["n"] / 40, 1)
                + min(val_stats["n"] / 30, 1)
            )

            if label != "neutral" and np.isfinite(train_stats["aligned_mean_pct"]):
                score += min(max(train_stats["aligned_mean_pct"], 0) / 2, 1)

            if label == "neutral" and np.isfinite(train_stats["aligned_mean_pct"]):
                score += min(max(-train_stats["aligned_mean_pct"], 0) / 1, 1)

            rows.append({
                "condition": " & ".join(condition_cols),
                "condition_cols_json": json.dumps(condition_cols),
                "rule_label": label,
                "rule_score": score,
                "rule_blocks": ", ".join(sorted({infer_rule_block(c) for c in condition_cols})),
                **{f"training_{k}": v for k, v in train_stats.items()},
                **{f"validation_{k}": v for k, v in val_stats.items()},
            })

    candidates_df = pd.DataFrame(rows)

    if candidates_df.empty:
        return candidates_df, candidates_df

    selected = candidates_df.sort_values(
        ["rule_label", "validation_accuracy_pct", "validation_n", "rule_score"],
        ascending=[True, False, False, False],
    )

    selected = selected.groupby("rule_label", group_keys=False).head(max_rules_per_label)

    selected = selected.sort_values(
        ["validation_accuracy_pct", "validation_n", "rule_score"],
        ascending=False,
    ).reset_index(drop=True)

    return candidates_df.sort_values("rule_score", ascending=False).reset_index(drop=True), selected


tune_train = model_df.loc[model_df["period"].eq("training")].copy()
tune_validation = model_df.loc[model_df["period"].eq("validation")].copy()
tune_train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()

base_spec = variant_specs[TUNE_VARIANT].copy()

tune_rows = []
tune_rule_frames = []

old_epochs = EPOCHS
EPOCHS = TUNE_EPOCHS

try:
    print("Validation tuning: no-logic baseline")

    no_logic_bundle = train_variant(
        "validation_grid_no_logic_baseline",
        base_spec,
        tune_train,
        use_logic=False,
        logic_weight=0.0,
        seed=TUNE_SEED,
        mine_rules=False,
    )

    no_logic_validation = score_variant(no_logic_bundle, tune_validation)
    no_logic_row = summary_row(
        "Neural classifier, Qwen + context interaction, no logic",
        "validation_tuning",
        no_logic_validation,
        bundle=no_logic_bundle,
    )

    no_logic_row.update({
        "rule_setting": "no_logic",
        "logic_weight": 0.0,
        "n_candidate_rules": 0,
        "n_selected_rules": 0,
        "min_training_events": np.nan,
        "min_validation_events": np.nan,
        "min_training_accuracy": np.nan,
        "min_validation_accuracy": np.nan,
        "selection_score": no_logic_row["accuracy_pct"] + max(no_logic_row["aligned_mean_pct"], 0),
    })

    tune_rows.append(no_logic_row)

    for rule_cfg in RULE_FILTER_GRID:
        print("Validation tuning rules:", rule_cfg["rule_setting"])

        candidate_rules, selected_rules = mine_validated_rules_with_thresholds(
            tune_train_val,
            base_spec["feature_cols"],
            base_spec["return_col"],
            rule_blocks=base_spec.get("rule_blocks"),
            min_training_events=rule_cfg["min_training_events"],
            min_validation_events=rule_cfg["min_validation_events"],
            min_training_accuracy=rule_cfg["min_training_accuracy"],
            min_validation_accuracy=rule_cfg["min_validation_accuracy"],
            max_pair_features=base_spec.get("max_pair_features", 36),
            max_rules_per_label=rule_cfg["max_rules_per_label"],
        )

        if len(selected_rules):
            tmp_rules = selected_rules.copy()
            tmp_rules["rule_setting"] = rule_cfg["rule_setting"]
            tune_rule_frames.append(tmp_rules)

        print(
            "candidate rules:",
            len(candidate_rules),
            "selected rules:",
            len(selected_rules),
        )

        tuned_spec = base_spec.copy()
        tuned_spec["selected_rules"] = selected_rules.copy()

        for logic_weight in [w for w in LOGIC_WEIGHT_GRID if w > 0]:
            print(
                "  training",
                rule_cfg["rule_setting"],
                "logic_weight",
                logic_weight,
            )

            bundle = train_variant(
                f"validation_grid_{rule_cfg['rule_setting']}_lw_{str(logic_weight).replace('.', 'p')}",
                tuned_spec,
                tune_train,
                use_logic=True,
                logic_weight=logic_weight,
                seed=TUNE_SEED,
                mine_rules=False,
            )

            validation_scored = score_variant(bundle, tune_validation)
            row = summary_row(
                "LTN, article-event + context interaction",
                "validation_tuning",
                validation_scored,
                bundle=bundle,
            )

            row.update({
                "rule_setting": rule_cfg["rule_setting"],
                "logic_weight": logic_weight,
                "n_candidate_rules": len(candidate_rules),
                "n_selected_rules": len(selected_rules),
                "min_training_events": rule_cfg["min_training_events"],
                "min_validation_events": rule_cfg["min_validation_events"],
                "min_training_accuracy": rule_cfg["min_training_accuracy"],
                "min_validation_accuracy": rule_cfg["min_validation_accuracy"],
                "selection_score": row["accuracy_pct"] + max(row["aligned_mean_pct"], 0),
            })

            tune_rows.append(row)

finally:
    EPOCHS = old_epochs


ltn_validation_grid = pd.DataFrame(tune_rows)

ltn_validation_grid_sorted = (
    ltn_validation_grid
    .sort_values(
        ["selection_score", "accuracy_pct", "aligned_mean_pct", "n_selected_rules"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(ltn_validation_grid_sorted)

if tune_rule_frames:
    ltn_validation_grid_selected_rules = pd.concat(tune_rule_frames, ignore_index=True)
    display(
        ltn_validation_grid_selected_rules
        .groupby(["rule_setting", "rule_label"])
        .agg(
            rules=("condition", "count"),
            validation_n_mean=("validation_n", "mean"),
            validation_accuracy_mean=("validation_accuracy_pct", "mean"),
            training_accuracy_mean=("training_accuracy_pct", "mean"),
        )
        .reset_index()
    )

best_ltn_config = ltn_validation_grid_sorted.iloc[0].to_dict()

print("Best validation-selected config:")
for key in [
    "variant",
    "rule_setting",
    "logic_weight",
    "accuracy_pct",
    "aligned_mean_pct",
    "selection_score",
    "n_selected_rules",
    "min_training_events",
    "min_validation_events",
    "min_training_accuracy",
    "min_validation_accuracy",
]:
    print(f"{key}: {best_ltn_config.get(key)}")

Validation tuning: no-logic baseline
Validation tuning rules: current_like
candidate rules: 91 selected rules: 24
  training current_like logic_weight 0.01
  training current_like logic_weight 0.03
  training current_like logic_weight 0.05
  training current_like logic_weight 0.1
  training current_like logic_weight 0.2
  training current_like logic_weight 0.3
  training current_like logic_weight 0.5
  training current_like logic_weight 0.75
  training current_like logic_weight 1.0
Validation tuning rules: moderate_support
candidate rules: 71 selected rules: 20
  training moderate_support logic_weight 0.01
  training moderate_support logic_weight 0.03
  training moderate_support logic_weight 0.05
  training moderate_support logic_weight 0.1
  training moderate_support logic_weight 0.2
  training moderate_support logic_weight 0.3
  training moderate_support logic_weight 0.5
  training moderate_support logic_weight 0.75
  training moderate_support logic_weight 1.0
Validation tuning rules

,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,mean_max_class_score,n_independent_clusters,aligned_mean_ci_low_pct,aligned_mean_ci_high_pct,aligned_bootstrap_p_value,use_logic,logic_weight,n_selected_rules,rule_setting,n_candidate_rules,min_training_events,min_validation_events,min_training_accuracy,min_validation_accuracy,selection_score
0,"LTN, article-event + context interaction",validation_tuning,1966,54.272635,-1.141053,0.653246,5.740902e-15,51.017294,45.218718,3.763988,0.725828,0.852574,254,0.240393,1.065796,0.0010,True,0.75,16,strict_accuracy,28,10.0,8.0,65.0,70.0,54.925880
1,"LTN, article-event + context interaction",validation_tuning,1966,54.272635,-1.141053,0.653246,5.740902e-15,51.017294,45.218718,3.763988,0.725828,0.852574,254,0.240393,1.065796,0.0010,True,0.75,16,high_support_strict,28,15.0,10.0,65.0,70.0,54.925880
2,"LTN, article-event + context interaction",validation_tuning,1966,52.390641,-1.141053,0.553300,1.618534e-11,49.898271,44.964395,5.137335,0.732814,0.854469,253,0.069840,1.008377,0.0142,True,0.50,24,current_like,91,6.0,5.0,60.0,60.0,52.943941
3,"LTN, article-event + context interaction",validation_tuning,1966,52.441506,-1.141053,0.482938,5.865741e-10,49.593082,46.490336,3.916582,0.709086,0.842809,255,-0.010557,0.949816,0.0270,True,1.00,16,strict_accuracy,28,10.0,8.0,65.0,70.0,52.924444
4,"LTN, article-event + context interaction",validation_tuning,1966,52.441506,-1.141053,0.482938,5.865741e-10,49.593082,46.490336,3.916582,0.709086,0.842809,255,-0.010557,0.949816,0.0270,True,1.00,16,high_support_strict,28,15.0,10.0,65.0,70.0,52.924444
5,"LTN, article-event + context interaction",validation_tuning,1966,52.034588,-1.141053,0.558192,3.529676e-10,48.626653,45.676501,5.696846,0.746550,0.861557,254,0.111676,0.991764,0.0074,True,0.50,16,strict_accuracy,28,10.0,8.0,65.0,70.0,52.592780
6,"LTN, article-event + context interaction",validation_tuning,1966,52.034588,-1.141053,0.558192,3.529676e-10,48.626653,45.676501,5.696846,0.746550,0.861557,254,0.111676,0.991764,0.0074,True,0.50,16,high_support_strict,28,15.0,10.0,65.0,70.0,52.592780
7,"LTN, article-event + context interaction",validation_tuning,1966,51.780264,-1.141053,0.717858,3.047363e-10,50.101729,44.913530,4.984741,0.750221,0.866156,254,0.275646,1.154530,0.0008,True,0.10,20,moderate_support,71,10.0,8.0,60.0,65.0,52.498122
8,"LTN, article-event + context interaction",validation_tuning,1966,51.983723,-1.141053,0.444602,5.313518e-10,52.797558,41.353001,5.849440,0.739027,0.858949,252,-0.088293,0.944092,0.0506,True,0.30,24,current_like,91,6.0,5.0,60.0,60.0,52.428325
9,"LTN, article-event + context interaction",validation_tuning,1966,51.831129,-1.141053,0.586015,1.466354e-11,47.965412,45.829095,6.205493,0.728403,0.851379,254,0.145066,1.025994,0.0042,True,0.05,20,moderate_support,71,10.0,8.0,60.0,65.0,52.417144


,rule_setting,rule_label,rules,validation_n_mean,validation_accuracy_mean,training_accuracy_mean
0,current_like,bearish,12,49.583333,85.963723,68.455492
1,current_like,bullish,12,61.750000,100.000000,68.133488
2,high_support_strict,bearish,8,63.125000,87.089525,71.341022
3,high_support_strict,bullish,8,49.000000,100.000000,76.148026
4,moderate_support,bearish,10,52.900000,88.004953,69.223010
5,moderate_support,bullish,10,68.700000,100.000000,68.481404
6,strict_accuracy,bearish,8,63.125000,87.089525,71.341022
7,strict_accuracy,bullish,8,49.000000,100.000000,76.148026


Best validation-selected config:
variant: LTN, article-event + context interaction
rule_setting: strict_accuracy
logic_weight: 0.75
accuracy_pct: 54.272634791454735
aligned_mean_pct: 0.6532456029587863
selection_score: 54.92588039441352
n_selected_rules: 16
min_training_events: 10.0
min_validation_events: 8.0
min_training_accuracy: 65.0
min_validation_accuracy: 70.0


## 10. Single-Seed LTN Versus No-LTN Ablation

This section compares the LTN Qwen-plus-context model with an equivalent neural classifier using the same inputs but no logic penalty.

In [15]:
train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()
test = model_df.loc[model_df["period"].eq("test")].copy()

ablation_base_spec = variant_specs[PRIMARY_VARIANT]


candidate_rules, selected_rules = mine_validated_rules(
    train_val,
    ablation_base_spec["feature_cols"],
    ablation_base_spec["return_col"],
    rule_blocks=ablation_base_spec.get("rule_blocks"),
    max_pair_features=ablation_base_spec.get("max_pair_features", 36),
    max_rules_per_label=ablation_base_spec.get("max_rules_per_label", 12),
)

ablation_ltn_spec = ablation_base_spec.copy()
ablation_ltn_spec["selected_rules"] = selected_rules.copy()


ltn_bundle = train_variant(
    "ltn_qwen_context_interaction_ablation",
    ablation_ltn_spec,
    train_val,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

ltn_scored = score_variant(ltn_bundle, test)


no_logic_bundle = train_variant(
    "no_logic_qwen_context_interaction_ablation",
    ablation_base_spec,
    train_val,
    use_logic=False,
    logic_weight=0.0,
    seed=SEED,
    mine_rules=False,
)

no_logic_scored = score_variant(no_logic_bundle, test)


ltn_ablation_summary = pd.DataFrame([
    summary_row(
        "LTN, article-event + context interaction",
        "heldout_test",
        ltn_scored,
        bundle=ltn_bundle,
    ),
    summary_row(
        "Neural classifier, Qwen + context interaction, no logic",
        "heldout_test",
        no_logic_scored,
        bundle=no_logic_bundle,
    ),
])

display(ltn_ablation_summary)


if SAVE_OUTPUTS:
    ltn_bundle["history"].to_csv(
        OUTPUT_DIR / "ltn_qwen_context_interaction_ablation_training_history.csv",
        index=False,
    )

    no_logic_bundle["history"].to_csv(
        OUTPUT_DIR / "no_logic_qwen_context_interaction_ablation_training_history.csv",
        index=False,
    )

    ltn_scored.to_csv(
        OUTPUT_DIR / "ltn_qwen_context_interaction_ablation_test_predictions.csv",
        index=False,
    )

    no_logic_scored.to_csv(
        OUTPUT_DIR / "no_logic_qwen_context_interaction_ablation_test_predictions.csv",
        index=False,
    )

    ltn_ablation_summary.to_csv(
        OUTPUT_DIR / "ltn_ablation_qwen_context_interaction_summary.csv",
        index=False,
    )

,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,mean_max_class_score,n_independent_clusters,aligned_mean_ci_low_pct,aligned_mean_ci_high_pct,aligned_bootstrap_p_value,use_logic,logic_weight,n_selected_rules
0,"LTN, article-event + context interaction",heldout_test,2775,42.450450,1.09779,0.213052,0.807607,58.450450,33.90991,7.639640,0.795790,0.890119,345,-0.339950,0.779916,0.2220,True,0.3,24
1,"Neural classifier, Qwen + context interaction,...",heldout_test,2775,41.657658,1.09779,0.322480,0.992400,52.720721,39.81982,7.459459,0.769762,0.875590,342,-0.260896,0.928972,0.1388,False,0.0,0


## 11. Repeated-Seed LTN Versus No-LTN Ablation

This section repeats the main LTN and no-logic comparison across multiple seeds to assess whether the result is stable.

In [16]:
seed_rows = []
seed_rule_rows = []

train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()
test = model_df.loc[model_df["period"].eq("test")].copy()

base_spec = variant_specs[PRIMARY_VARIANT]

for seed in ROBUST_SEEDS:
    print(f"Main repeated-seed ablation: seed {seed}")

    candidate_rules, selected_rules = mine_validated_rules(
        train_val,
        base_spec["feature_cols"],
        base_spec["return_col"],
        rule_blocks=base_spec.get("rule_blocks"),
        max_pair_features=base_spec.get("max_pair_features", 36),
        max_rules_per_label=base_spec.get("max_rules_per_label", 12),
    )

    ltn_spec = base_spec.copy()
    ltn_spec["selected_rules"] = selected_rules.copy()

    seed_rule_rows.append({
        "seed": seed,
        "model": "LTN, article-event + context interaction",
        "rule_blocks": ", ".join(base_spec.get("rule_blocks", [])),
        "n_candidate_rules": len(candidate_rules),
        "n_selected_rules": len(selected_rules),
        "validation_n_min": selected_rules["validation_n"].min() if len(selected_rules) else np.nan,
        "validation_n_median": selected_rules["validation_n"].median() if len(selected_rules) else np.nan,
        "validation_n_max": selected_rules["validation_n"].max() if len(selected_rules) else np.nan,
        "validation_accuracy_mean": selected_rules["validation_accuracy_pct"].mean() if len(selected_rules) else np.nan,
    })

    ltn_bundle = train_variant(
        f"ltn_qwen_context_interaction_seed_{seed}",
        ltn_spec,
        train_val,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    ltn_scored = score_variant(ltn_bundle, test)

    seed_rows.append({
        "seed": seed,
        "model": "LTN, article-event + context interaction",
        **summary_row("LTN, article-event + context interaction", "heldout_test", ltn_scored, bundle=ltn_bundle),
    })

    no_logic_bundle = train_variant(
        f"no_logic_qwen_context_interaction_seed_{seed}",
        base_spec,
        train_val,
        use_logic=False,
        logic_weight=0.0,
        seed=seed,
        mine_rules=False,
    )

    no_logic_scored = score_variant(no_logic_bundle, test)

    seed_rows.append({
        "seed": seed,
        "model": "Neural classifier, Qwen + context interaction, no logic",
        **summary_row(
            "Neural classifier, Qwen + context interaction, no logic",
            "heldout_test",
            no_logic_scored,
            bundle=no_logic_bundle,
        ),
    })


seed_results = pd.DataFrame(seed_rows)
seed_rule_summary = pd.DataFrame(seed_rule_rows)

seed_summary = (
    seed_results
    .groupby("model")
    .agg(
        runs=("seed", "nunique"),
        accuracy_mean=("accuracy_pct", "mean"),
        accuracy_std=("accuracy_pct", "std"),
        aligned_mean_return_mean=("aligned_mean_pct", "mean"),
        aligned_mean_return_std=("aligned_mean_pct", "std"),
        confidence_margin_mean=("mean_confidence_margin", "mean"),
        confidence_margin_std=("mean_confidence_margin", "std"),
        selected_rules_mean=("n_selected_rules", "mean"),
        selected_rules_min=("n_selected_rules", "min"),
        selected_rules_max=("n_selected_rules", "max"),
    )
    .reset_index()
)

display(seed_rule_summary)
display(seed_results)
display(seed_summary)

Main repeated-seed ablation: seed 1
Main repeated-seed ablation: seed 2
Main repeated-seed ablation: seed 3
Main repeated-seed ablation: seed 4
Main repeated-seed ablation: seed 5
Main repeated-seed ablation: seed 7
Main repeated-seed ablation: seed 11
Main repeated-seed ablation: seed 13
Main repeated-seed ablation: seed 17
Main repeated-seed ablation: seed 19


,seed,model,rule_blocks,n_candidate_rules,n_selected_rules,validation_n_min,validation_n_median,validation_n_max,validation_accuracy_mean
0,1,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1536,24,12,44.0,126,92.981862
1,2,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1536,24,12,44.0,126,92.981862
2,3,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1536,24,12,44.0,126,92.981862
3,4,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1536,24,12,44.0,126,92.981862
4,5,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1536,24,12,44.0,126,92.981862
5,7,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1536,24,12,44.0,126,92.981862
6,11,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1536,24,12,44.0,126,92.981862
7,13,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1536,24,12,44.0,126,92.981862
8,17,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1536,24,12,44.0,126,92.981862
9,19,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1536,24,12,44.0,126,92.981862


,seed,model,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,mean_max_class_score,n_independent_clusters,aligned_mean_ci_low_pct,aligned_mean_ci_high_pct,aligned_bootstrap_p_value,use_logic,logic_weight,n_selected_rules
0,1,"LTN, article-event + context interaction","LTN, article-event + context interaction",heldout_test,2775,41.801802,1.09779,0.120214,0.965312,55.603604,36.900901,7.495495,0.807801,0.898596,346,-0.448442,0.713777,0.3548,True,0.3,24
1,1,"Neural classifier, Qwen + context interaction,...","Neural classifier, Qwen + context interaction,...",heldout_test,2775,45.261261,1.09779,0.481110,0.347364,57.333333,36.252252,6.414414,0.819315,0.904524,344,-0.092831,1.060054,0.0492,False,0.0,0
2,2,"LTN, article-event + context interaction","LTN, article-event + context interaction",heldout_test,2775,42.342342,1.09779,0.311553,0.640384,53.765766,36.576577,9.657658,0.802730,0.895042,341,-0.216592,0.871436,0.1306,True,0.3,24
3,2,"Neural classifier, Qwen + context interaction,...","Neural classifier, Qwen + context interaction,...",heldout_test,2775,39.747748,1.09779,-0.008428,0.996803,53.153153,35.207207,11.639640,0.800440,0.893919,344,-0.562844,0.573460,0.5114,False,0.0,0
4,3,"LTN, article-event + context interaction","LTN, article-event + context interaction",heldout_test,2775,42.018018,1.09779,0.291190,0.923733,55.531532,35.747748,8.720721,0.811604,0.899915,344,-0.237885,0.840842,0.1474,True,0.3,24
5,3,"Neural classifier, Qwen + context interaction,...","Neural classifier, Qwen + context interaction,...",heldout_test,2775,43.711712,1.09779,0.359630,0.346576,53.189189,39.387387,7.423423,0.808767,0.898345,343,-0.166678,0.899788,0.0920,False,0.0,0
6,4,"LTN, article-event + context interaction","LTN, article-event + context interaction",heldout_test,2775,40.972973,1.09779,0.281059,0.995239,54.198198,36.432432,9.369369,0.782647,0.883786,341,-0.291624,0.869601,0.1766,True,0.3,24
7,4,"Neural classifier, Qwen + context interaction,...","Neural classifier, Qwen + context interaction,...",heldout_test,2775,41.729730,1.09779,0.264021,0.580164,50.306306,37.729730,11.963964,0.786398,0.887041,344,-0.285860,0.838945,0.1752,False,0.0,0
8,5,"LTN, article-event + context interaction","LTN, article-event + context interaction",heldout_test,2775,43.747748,1.09779,0.430093,0.044747,50.846847,39.459459,9.693694,0.791089,0.887683,342,-0.143698,1.000121,0.0744,True,0.3,24
9,5,"Neural classifier, Qwen + context interaction,...","Neural classifier, Qwen + context interaction,...",heldout_test,2775,40.684685,1.09779,0.261581,0.942883,52.612613,37.549550,9.837838,0.801504,0.894221,343,-0.326814,0.858936,0.1886,False,0.0,0


,model,runs,accuracy_mean,accuracy_std,aligned_mean_return_mean,aligned_mean_return_std,confidence_margin_mean,confidence_margin_std,selected_rules_mean,selected_rules_min,selected_rules_max
0,"LTN, article-event + context interaction",10,42.371171,0.914776,0.287756,0.112756,0.796932,0.011095,24.0,24,24
1,"Neural classifier, Qwen + context interaction,...",10,42.526126,1.635792,0.286509,0.132038,0.802609,0.015103,0.0,0,0


**Reading this table:** across the 10 seeds, the LTN variant (42.4% $\pm$ 0.9% accuracy, 0.29% aligned mean return) and the no-logic neural network (42.5% $\pm$ 1.6%, 0.29% aligned mean return) are statistically indistinguishable -- the aligned mean return is identical to two decimal places. Section 15 repeats this ablation on the longer-window robustness split; Section 16 discusses both together.

### Rule Generalisation Check

The cells above select rules using training and validation accuracy thresholds. Clearing both thresholds does not guarantee a rule generalises -- with a large candidate pool and small per-condition samples, some rules will clear both by chance. This section applies the frozen Section 9 rules directly to the held-out test set, so training and validation accuracy can be compared row-by-row against genuine out-of-sample accuracy for the same rule.

In [17]:
# Show full-classifier rules and how they perform on the test set

import json
import numpy as np
import pandas as pd

RULE_MATCH_THRESHOLD = ANTECEDENT_THRESHOLD if "ANTECEDENT_THRESHOLD" in globals() else 0.50

rows = []

for variant_name, spec in frozen_variant_specs.items():
    rules = spec.get("selected_rules", pd.DataFrame()).copy()

    if rules.empty:
        continue

    test_eval = score_variant(trained_final[variant_name], test)

    for _, rule in rules.iterrows():
        condition_cols = json.loads(rule["condition_cols_json"])
        rule_label = rule["rule_label"]

        mask = np.ones(len(test_eval), dtype=bool)

        for col in condition_cols:
            mask &= (
                pd.to_numeric(test_eval[col], errors="coerce")
                .fillna(0)
                .ge(RULE_MATCH_THRESHOLD)
                .to_numpy()
            )

        matched = test_eval.loc[mask].copy()

        if len(matched) == 0:
            continue

        rows.append({
            "variant": variant_name,
            "rule": f"{rule['condition']} -> {rule_label}",
            "rule_label": rule_label,

            "training_matches": rule.get("training_n", np.nan),
            "training_accuracy_pct": rule.get("training_accuracy_pct", np.nan),

            "validation_matches": rule.get("validation_n", np.nan),
            "validation_accuracy_pct": rule.get("validation_accuracy_pct", np.nan),

            "test_matches": len(matched),
            "test_accuracy_pct": 100 * matched["target_label"].eq(rule_label).mean(),

            "test_bearish": int(matched["target_label"].eq("bearish").sum()),
            "test_bullish": int(matched["target_label"].eq("bullish").sum()),
            "test_neutral": int(matched["target_label"].eq("neutral").sum()),
        })

full_classifier_rules_on_test = pd.DataFrame(rows)

display(
    full_classifier_rules_on_test
    .sort_values(
        ["variant", "test_accuracy_pct", "test_matches"],
        ascending=[True, False, False],
    )
    .reset_index(drop=True)
)

,variant,rule,rule_label,training_matches,training_accuracy_pct,validation_matches,validation_accuracy_pct,test_matches,test_accuracy_pct,test_bearish,test_bullish,test_neutral
0,full_ltn_context_only,ticker_prior_down_5d & quiet_ticker_5d -> bullish,bullish,664,98.945783,66,62.121212,36,72.222222,10,26,0
1,full_ltn_context_only,market_prior_down_5d & relative_prior_up_5d ->...,bearish,340,65.294118,27,100.000000,155,67.741935,105,44,6
2,full_ltn_context_only,market_prior_down_5d & ticker_prior_down_5d ->...,bullish,1183,62.806424,249,69.879518,295,41.355932,131,122,42
3,full_ltn_context_only,hour_sin_01 & is_monday -> bearish,bearish,1452,62.121212,222,63.963964,625,39.040000,244,356,25
4,full_ltn_context_only,market_prior_down_5d & quiet_ticker_5d -> bullish,bullish,54,70.370370,105,64.761905,125,36.000000,80,45,0
...,...,...,...,...,...,...,...,...,...,...,...,...
63,full_ltn_qwen_plus_context,ticker_prior_up_5d & event_type_regulatory_leg...,bearish,238,64.705882,33,75.757576,88,25.000000,22,55,11
64,full_ltn_qwen_plus_context,log1p_same_day_article_event_count & event_typ...,bullish,1405,83.629893,14,100.000000,59,23.728814,43,14,2
65,full_ltn_qwen_plus_context,relative_prior_up_5d & volatile_ticker_5d -> b...,bearish,129,74.418605,9,100.000000,42,21.428571,9,33,0
66,full_ltn_qwen_plus_context,ticker_prior_up_5d & volatile_ticker_5d -> bea...,bearish,119,72.268908,8,100.000000,42,21.428571,9,33,0


**Reading this table:** several LLM-derived rules collapse sharply from validation to test -- for example a rule conditioned on `event_type_earnings_guidance` drops from 76.2% validation accuracy (n=126) to 27.0% test accuracy (n=278) for a bearish call, worse than chance. Rules built only from price-context features tend to hold up better, e.g. `market_prior_down_5d & relative_prior_up_5d` (bearish): 100% on a small validation sample (n=27) to a believable 67.7% on 155 test rows. This matches the Section 9 finding that the context-only variant generalises better than variants that include LLM-derived event features: the categorical event-type/subtype feature space appears more prone to small-sample overfitting than the continuous price features.

## 12. Selective Classifier

This section evaluates whether the LTN performs better when it abstains from low-confidence cases and keeps only high-confidence bullish or bearish predictions.

In [18]:

TRUE_SELECTIVE_FILTER_VARIANT = PRIMARY_VARIANT
TRUE_SELECTIVE_SIGNAL_LABELS = ["bullish", "bearish"]
TRUE_SELECTIVE_QUANTILES = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90]

if TRUE_SELECTIVE_FILTER_VARIANT not in frozen_variant_specs:
    raise KeyError(
        f"{TRUE_SELECTIVE_FILTER_VARIANT!r} was not found in frozen_variant_specs. "
        f"Available variants: {list(frozen_variant_specs)}"
    )


def apply_true_selective_filter(scored, bullish_threshold, bearish_threshold):
    out = scored.copy()

    bullish_signal = (
        out["prediction"].eq("bullish")
        & out["score_bullish"].ge(bullish_threshold)
    )

    bearish_signal = (
        out["prediction"].eq("bearish")
        & out["score_bearish"].ge(bearish_threshold)
    )

    out["selective_prediction"] = "no_signal"
    out.loc[bullish_signal, "selective_prediction"] = "bullish"
    out.loc[bearish_signal, "selective_prediction"] = "bearish"

    out["has_signal"] = out["selective_prediction"].isin(TRUE_SELECTIVE_SIGNAL_LABELS)

    return out











































def true_selective_subset_stats(
    df,
    variant,
    period,
    signal_label=None,
    threshold_quantile=None,
):
    if signal_label is None:
        subset = df.loc[df["has_signal"]].copy()
        label_name = "joint"
    else:
        subset = df.loc[df["selective_prediction"].eq(signal_label)].copy()
        label_name = signal_label

    row = {
        "variant": variant,
        "period": period,
        "signal": label_name,
        "threshold_quantile": threshold_quantile,
        "n_total": int(len(df)),
        "n_signal": int(len(subset)),
        "coverage_pct": float(100 * len(subset) / len(df)) if len(df) else np.nan,
        "neutral_events_in_total": int(df["target_label"].eq("neutral").sum()),
        "neutral_event_rate_pct": float(100 * df["target_label"].eq("neutral").mean()) if len(df) else np.nan,
    }

    if subset.empty:
        row.update({
            "accuracy_pct": np.nan,
            "aligned_mean_pct": np.nan,
            "aligned_signflip_p": np.nan,
            "mean_simple_return_pct": np.nan,
            "selected_true_bullish_n": 0,
            "selected_true_bearish_n": 0,
            "selected_true_neutral_n": 0,
            "n_independent_clusters": 0,
            "aligned_mean_ci_low_pct": np.nan,
            "aligned_mean_ci_high_pct": np.nan,
            "aligned_bootstrap_p_value": np.nan,
        })
        return row

    ret = pd.to_numeric(subset["simple_return"], errors="coerce")
    pred = subset["selective_prediction"]

    signed = np.select(
        [pred.eq("bullish"), pred.eq("bearish")],
        [ret, -ret],
        default=np.nan,
    )
    signed = pd.Series(signed, index=subset.index).dropna()

    if len(signed) >= 3:
        p = binomtest(
            int((signed > 0).sum()),
            len(signed),
            0.5,
            alternative="greater",
        ).pvalue
        aligned_mean = float(100 * signed.mean())
    else:
        p = np.nan
        aligned_mean = np.nan

    # Pass the prediction column name explicitly instead of renaming into
    # "prediction" - subset already has its own "prediction" column from
    # score_variant, so a rename would collide with it.
    cluster_stats = cluster_bootstrap_aligned_stats(
        subset,
        prediction_col="selective_prediction",
    )

    row.update({
        "accuracy_pct": float(100 * subset["selective_prediction"].eq(subset["target_label"]).mean()),
        "aligned_mean_pct": aligned_mean,
        "aligned_signflip_p": float(p) if np.isfinite(p) else np.nan,
        "mean_simple_return_pct": float(100 * ret.mean()),
        "selected_true_bullish_n": int(subset["target_label"].eq("bullish").sum()),
        "selected_true_bearish_n": int(subset["target_label"].eq("bearish").sum()),
        "selected_true_neutral_n": int(subset["target_label"].eq("neutral").sum()),
        **cluster_stats,
    })

    return row

In [19]:

true_threshold_train = model_df.loc[model_df["period"].eq("training")].copy()
true_threshold_validation = model_df.loc[model_df["period"].eq("validation")].copy()

name = TRUE_SELECTIVE_FILTER_VARIANT
spec = frozen_variant_specs[name]

print("true selective threshold calibration", name)
print("training rows:", len(true_threshold_train))
print("validation rows:", len(true_threshold_validation))
display(true_threshold_validation["target_label"].value_counts().to_frame("validation_labels"))

true_threshold_bundle = train_variant(
    f"{name}_true_selective_threshold_calibration",
    spec,
    true_threshold_train,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

true_validation_scored = score_variant(
    true_threshold_bundle,
    true_threshold_validation,
)

true_threshold_grid_rows = []

for q in TRUE_SELECTIVE_QUANTILES:
    bullish_threshold = float(true_validation_scored["score_bullish"].quantile(q))
    bearish_threshold = float(true_validation_scored["score_bearish"].quantile(q))

    validation_q = apply_true_selective_filter(
        true_validation_scored,
        bullish_threshold,
        bearish_threshold,
    )

    for signal_label in ["bullish", "bearish", None]:
        true_threshold_grid_rows.append(
            true_selective_subset_stats(
                validation_q,
                variant=name,
                period="validation_true_selective_grid",
                signal_label=signal_label,
                threshold_quantile=q,
            )
        )

true_threshold_grid_summary = pd.DataFrame(true_threshold_grid_rows)

display(
    true_threshold_grid_summary
    .sort_values(["threshold_quantile", "signal"])
    .loc[:, [
        "variant",
        "period",
        "signal",
        "threshold_quantile",
        "n_total",
        "n_signal",
        "coverage_pct",
        "accuracy_pct",
        "aligned_mean_pct",
        "aligned_signflip_p",
        "selected_true_bullish_n",
        "selected_true_bearish_n",
        "selected_true_neutral_n",
        "neutral_events_in_total",
    ]]
)

true selective threshold calibration full_ltn_qwen_context_interaction
training rows: 7349
validation rows: 1966


,validation_labels
target_label,
bearish,1081
bullish,801
neutral,84


,variant,period,signal,threshold_quantile,n_total,n_signal,coverage_pct,accuracy_pct,aligned_mean_pct,aligned_signflip_p,selected_true_bullish_n,selected_true_bearish_n,selected_true_neutral_n,neutral_events_in_total
1,full_ltn_qwen_context_interaction,validation_true_selective_grid,bearish,0.50,1966,774,39.369278,65.503876,1.895527,8.635026e-20,253,507,14,84
0,full_ltn_qwen_context_interaction,validation_true_selective_grid,bullish,0.50,1966,983,50.000000,47.405900,-0.566704,2.034817e-01,466,458,59,84
2,full_ltn_qwen_context_interaction,validation_true_selective_grid,joint,0.50,1966,1757,89.369278,55.378486,0.517967,2.085775e-11,719,965,73,84
4,full_ltn_qwen_context_interaction,validation_true_selective_grid,bearish,0.60,1966,769,39.114954,65.929779,1.924688,2.343791e-20,249,507,13,84
3,full_ltn_qwen_context_interaction,validation_true_selective_grid,bullish,0.60,1966,787,40.030519,49.555273,-0.474178,1.931086e-02,390,346,51,84
5,full_ltn_qwen_context_interaction,validation_true_selective_grid,joint,0.60,1966,1556,79.145473,57.647815,0.711380,1.283159e-15,639,853,64,84
7,full_ltn_qwen_context_interaction,validation_true_selective_grid,bearish,0.70,1966,590,30.010173,68.135593,2.151035,1.642136e-20,177,402,11,84
6,full_ltn_qwen_context_interaction,validation_true_selective_grid,bullish,0.70,1966,590,30.010173,49.830508,-0.303573,7.535295e-03,294,252,44,84
8,full_ltn_qwen_context_interaction,validation_true_selective_grid,joint,0.70,1966,1180,60.020346,58.983051,0.923731,1.025123e-16,471,654,55,84
10,full_ltn_qwen_context_interaction,validation_true_selective_grid,bearish,0.75,1966,492,25.025432,69.105691,2.085818,1.343774e-18,144,340,8,84


In [20]:

TRUE_CONFIDENCE_THRESHOLD = 0.90

In [21]:

true_threshold_test = model_df.loc[model_df["period"].eq("test")].copy()
true_threshold_train_val = model_df.loc[
    model_df["period"].isin(["training", "validation"])
].copy()

name = TRUE_SELECTIVE_FILTER_VARIANT
spec = frozen_variant_specs[name]

bullish_threshold = float(
    true_validation_scored["score_bullish"].quantile(TRUE_CONFIDENCE_THRESHOLD)
)
bearish_threshold = float(
    true_validation_scored["score_bearish"].quantile(TRUE_CONFIDENCE_THRESHOLD)
)

true_manual_threshold_selection_summary = pd.DataFrame([{
    "variant": name,
    "confidence_threshold_quantile": TRUE_CONFIDENCE_THRESHOLD,
    "bullish_threshold": bullish_threshold,
    "bearish_threshold": bearish_threshold,
    "selection_mode": "manual_validation_quantile_true_selective",
    "n_selected_rules": len(spec["selected_rules"]),
}])

display(true_manual_threshold_selection_summary)

print("true selective final refit", name)
print("training+validation rows:", len(true_threshold_train_val))
print("test rows:", len(true_threshold_test))
display(true_threshold_test["target_label"].value_counts().to_frame("test_labels"))

true_final_bundle = train_variant(
    f"{name}_true_selective_final_refit",
    spec,
    true_threshold_train_val,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

true_test_scored = score_variant(true_final_bundle, true_threshold_test)

true_test_selective = apply_true_selective_filter(
    true_test_scored,
    bullish_threshold,
    bearish_threshold,
)

true_selective_test_rows = []

for signal_label in ["bullish", "bearish", None]:
    true_selective_test_rows.append(
        true_selective_subset_stats(
            true_test_selective,
            variant=name,
            period="test_true_selective_filter",
            signal_label=signal_label,
            threshold_quantile=TRUE_CONFIDENCE_THRESHOLD,
        )
    )

true_selective_high_confidence_summary = pd.DataFrame(true_selective_test_rows)

display(
    true_selective_high_confidence_summary.sort_values(
        ["signal", "accuracy_pct", "coverage_pct"],
        ascending=[True, False, False],
    )
)


true_selected_display_cols = [
    "publication_timestamp",
    "published_at",
    "datetime",
    "event_date",
    "ticker",
    "title",
    "headline",
    "summary",
    "target_label",
    "prediction",
    "selective_prediction",
    "simple_return",
    RETURN_COL if "RETURN_COL" in globals() else None,
    "score_bullish",
    "score_bearish",
    "score_neutral",
    "confidence_margin",
]

true_selected_display_cols = [
    col for col in true_selected_display_cols
    if col is not None and col in true_test_selective.columns
]

display(
    true_test_selective
    .loc[true_test_selective["has_signal"], true_selected_display_cols]
    .sort_values("confidence_margin", ascending=False)
)

,variant,confidence_threshold_quantile,bullish_threshold,bearish_threshold,selection_mode,n_selected_rules
0,full_ltn_qwen_context_interaction,0.9,1.0,0.999695,manual_validation_quantile_true_selective,24


true selective final refit full_ltn_qwen_context_interaction
training+validation rows: 9315
test rows: 2775


,test_labels
target_label,
bullish,1412
bearish,1108
neutral,255


,variant,period,signal,threshold_quantile,n_total,n_signal,coverage_pct,neutral_events_in_total,neutral_event_rate_pct,accuracy_pct,aligned_mean_pct,aligned_signflip_p,mean_simple_return_pct,selected_true_bullish_n,selected_true_bearish_n,selected_true_neutral_n,n_independent_clusters,aligned_mean_ci_low_pct,aligned_mean_ci_high_pct,aligned_bootstrap_p_value
1,full_ltn_qwen_context_interaction,test_true_selective_filter,bearish,0.9,2775,223,8.036036,255,9.189189,54.708520,0.075419,0.053905,-0.075419,90,122,11,64,-2.380588,1.977053,0.4988
0,full_ltn_qwen_context_interaction,test_true_selective_filter,bullish,0.9,2775,211,7.603604,255,9.189189,65.876777,3.076587,0.000001,3.076587,139,56,16,51,1.104970,5.205283,0.0000
2,full_ltn_qwen_context_interaction,test_true_selective_filter,joint,0.9,2775,434,15.639640,255,9.189189,60.138249,1.534512,0.000004,1.457008,229,178,27,111,-0.047017,3.003322,0.0304


,event_date,ticker,target_label,prediction,selective_prediction,simple_return,score_bullish,score_bearish,score_neutral,confidence_margin
12241,2026-05-22,TSLA,bullish,bullish,bullish,0.031041,1.000000,7.751881e-15,2.437642e-19,1.000000
12229,2026-05-07,TSLA,bullish,bullish,bullish,0.092741,1.000000,4.339720e-11,1.189526e-12,1.000000
12227,2026-05-07,TSLA,bullish,bullish,bullish,0.092741,1.000000,2.827754e-10,3.202496e-12,1.000000
12213,2026-04-28,TSLA,bullish,bullish,bullish,0.047587,1.000000,8.660566e-13,6.885484e-10,1.000000
1417,2026-03-05,AAPL,neutral,bullish,bullish,-0.000844,1.000000,1.983377e-12,2.275684e-08,1.000000
...,...,...,...,...,...,...,...,...,...,...
10372,2026-03-16,NVDA,bearish,bearish,bearish,-0.057228,0.000265,9.997217e-01,1.353273e-05,0.999457
7888,2026-04-23,MSFT,bullish,bearish,bearish,0.008699,0.000273,9.997259e-01,9.824887e-07,0.999453
3072,2026-03-09,AMZN,bearish,bearish,bearish,-0.013210,0.000274,9.997184e-01,7.624159e-06,0.999444
5286,2026-03-23,GOOGL,bearish,bearish,bearish,-0.092462,0.000277,9.996982e-01,2.498375e-05,0.999421


In [22]:

TRUE_ROBUST_THRESHOLD_SEEDS = [1, 2, 3, 4, 5, 7, 11, 13, 17, 19]

true_robust_threshold_rows = []
true_robust_threshold_rule_rows = []
true_robust_threshold_prediction_frames = []

true_threshold_train = model_df.loc[model_df["period"].eq("training")].copy()
true_threshold_validation = model_df.loc[model_df["period"].eq("validation")].copy()
true_threshold_train_val = model_df.loc[
    model_df["period"].isin(["training", "validation"])
].copy()
true_threshold_test = model_df.loc[model_df["period"].eq("test")].copy()

name = TRUE_SELECTIVE_FILTER_VARIANT
base_spec = variant_specs[name]

for seed in TRUE_ROBUST_THRESHOLD_SEEDS:
    print("true selective-filter robustness seed", seed)

    candidate_rules, selected_rules = mine_validated_rules(
        true_threshold_train_val,
        base_spec["feature_cols"],
        base_spec["return_col"],
        rule_blocks=base_spec.get("rule_blocks"),
        max_pair_features=base_spec.get("max_pair_features", 36),
        max_rules_per_label=base_spec.get("max_rules_per_label", 12),
    )

    true_robust_threshold_rule_rows.append({
        "seed": seed,
        "variant": name,
        "n_candidate_rules": len(candidate_rules),
        "n_selected_rules": len(selected_rules),
        "validation_n_min": selected_rules["validation_n"].min() if len(selected_rules) else np.nan,
        "validation_n_median": selected_rules["validation_n"].median() if len(selected_rules) else np.nan,
        "validation_n_max": selected_rules["validation_n"].max() if len(selected_rules) else np.nan,
        "validation_accuracy_mean": selected_rules["validation_accuracy_pct"].mean() if len(selected_rules) else np.nan,
    })

    robust_spec = base_spec.copy()
    robust_spec["selected_rules"] = selected_rules.copy()

    calibration_bundle = train_variant(
        f"{name}_true_selective_threshold_calibration_seed_{seed}",
        robust_spec,
        true_threshold_train,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    validation_scored = score_variant(calibration_bundle, true_threshold_validation)

    bullish_threshold = float(
        validation_scored["score_bullish"].quantile(TRUE_CONFIDENCE_THRESHOLD)
    )
    bearish_threshold = float(
        validation_scored["score_bearish"].quantile(TRUE_CONFIDENCE_THRESHOLD)
    )

    final_bundle = train_variant(
        f"{name}_true_selective_final_refit_seed_{seed}",
        robust_spec,
        true_threshold_train_val,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    test_scored = score_variant(final_bundle, true_threshold_test)

    test_selective = apply_true_selective_filter(
        test_scored,
        bullish_threshold,
        bearish_threshold,
    )

    true_robust_threshold_prediction_frames.append(
        test_selective.assign(
            seed=seed,
            variant=name,
            period="test_true_selective_repeated_seed",
            confidence_threshold_quantile=TRUE_CONFIDENCE_THRESHOLD,
            bullish_threshold=bullish_threshold,
            bearish_threshold=bearish_threshold,
            n_selected_rules=len(selected_rules),
        )
    )

    for signal_label in ["bullish", "bearish", None]:
        row = true_selective_subset_stats(
            test_selective,
            variant=name,
            period="test_true_selective_repeated_seed",
            signal_label=signal_label,
            threshold_quantile=TRUE_CONFIDENCE_THRESHOLD,
        )
        row.update({
            "seed": seed,
            "bullish_threshold": bullish_threshold,
            "bearish_threshold": bearish_threshold,
            "n_selected_rules": len(selected_rules),
        })
        true_robust_threshold_rows.append(row)

true_robust_threshold_seed_summary = pd.DataFrame(true_robust_threshold_rows)
true_robust_threshold_rule_summary = pd.DataFrame(true_robust_threshold_rule_rows)
true_robust_threshold_predictions = pd.concat(
    true_robust_threshold_prediction_frames,
    ignore_index=True,
)

display(
    true_robust_threshold_seed_summary.sort_values(
        ["seed", "signal"]
    )
)

true_robust_threshold_summary = (
    true_robust_threshold_seed_summary
    .groupby(["variant", "signal"])
    .agg(
        n_seeds=("seed", "nunique"),
        n_total=("n_total", "first"),
        n_signal_mean=("n_signal", "mean"),
        n_signal_min=("n_signal", "min"),
        n_signal_max=("n_signal", "max"),
        coverage_mean=("coverage_pct", "mean"),
        coverage_std=("coverage_pct", "std"),
        accuracy_mean=("accuracy_pct", "mean"),
        accuracy_std=("accuracy_pct", "std"),
        accuracy_min=("accuracy_pct", "min"),
        accuracy_max=("accuracy_pct", "max"),
        aligned_mean_mean=("aligned_mean_pct", "mean"),
        aligned_mean_std=("aligned_mean_pct", "std"),
        p_value_median=("aligned_signflip_p", "median"),
        selected_rules_mean=("n_selected_rules", "mean"),
        selected_rules_min=("n_selected_rules", "min"),
        selected_rules_max=("n_selected_rules", "max"),
        selected_true_bullish_mean=("selected_true_bullish_n", "mean"),
        selected_true_bearish_mean=("selected_true_bearish_n", "mean"),
        selected_true_neutral_mean=("selected_true_neutral_n", "mean"),
    )
    .reset_index()
)

display(true_robust_threshold_rule_summary)
display(true_robust_threshold_summary)

true selective-filter robustness seed 1
true selective-filter robustness seed 2
true selective-filter robustness seed 3
true selective-filter robustness seed 4
true selective-filter robustness seed 5
true selective-filter robustness seed 7
true selective-filter robustness seed 11
true selective-filter robustness seed 13
true selective-filter robustness seed 17
true selective-filter robustness seed 19


,variant,period,signal,threshold_quantile,n_total,n_signal,coverage_pct,neutral_events_in_total,neutral_event_rate_pct,accuracy_pct,aligned_mean_pct,aligned_signflip_p,mean_simple_return_pct,selected_true_bullish_n,selected_true_bearish_n,selected_true_neutral_n,n_independent_clusters,aligned_mean_ci_low_pct,aligned_mean_ci_high_pct,aligned_bootstrap_p_value,seed,bullish_threshold,bearish_threshold,n_selected_rules
1,full_ltn_qwen_context_interaction,test_true_selective_repeated_seed,bearish,0.9,2775,213,7.675676,255,9.189189,50.234742,-0.114066,4.455214e-01,0.114066,92,107,14,61,-2.219136,1.720919,0.5560,1,1.000000,0.999938,24
0,full_ltn_qwen_context_interaction,test_true_selective_repeated_seed,bullish,0.9,2775,235,8.468468,255,9.189189,61.702128,2.567321,7.356799e-06,2.567321,145,68,22,60,0.738182,4.386143,0.0018,1,1.000000,0.999938,24
2,full_ltn_qwen_context_interaction,test_true_selective_repeated_seed,joint,0.9,2775,448,16.144144,255,9.189189,56.250000,1.292465,5.457159e-04,1.400930,237,175,36,114,-0.103651,2.590143,0.0354,1,1.000000,0.999938,24
4,full_ltn_qwen_context_interaction,test_true_selective_repeated_seed,bearish,0.9,2775,193,6.954955,255,9.189189,53.886010,0.228461,7.489170e-02,-0.228461,80,104,9,46,-2.615081,2.281736,0.4720,2,1.000000,0.999992,24
3,full_ltn_qwen_context_interaction,test_true_selective_repeated_seed,bullish,0.9,2775,260,9.369369,255,9.189189,61.153846,2.758234,8.403455e-06,2.758234,159,81,20,63,1.045704,4.611129,0.0012,2,1.000000,0.999992,24
5,full_ltn_qwen_context_interaction,test_true_selective_repeated_seed,joint,0.9,2775,453,16.324324,255,9.189189,58.057395,1.680428,1.108221e-05,1.485757,239,185,29,105,0.262563,3.004101,0.0112,2,1.000000,0.999992,24
7,full_ltn_qwen_context_interaction,test_true_selective_repeated_seed,bearish,0.9,2775,251,9.045045,255,9.189189,50.996016,0.064713,3.068392e-01,-0.064713,106,128,17,63,-2.432543,2.076638,0.5098,3,1.000000,0.999878,24
6,full_ltn_qwen_context_interaction,test_true_selective_repeated_seed,bullish,0.9,2775,274,9.873874,255,9.189189,58.029197,2.206014,4.296259e-04,2.206014,159,92,23,59,0.488248,3.912610,0.0050,3,1.000000,0.999878,24
8,full_ltn_qwen_context_interaction,test_true_selective_repeated_seed,joint,0.9,2775,525,18.918919,255,9.189189,54.666667,1.182268,2.585206e-03,1.120390,265,220,40,114,-0.096350,2.331348,0.0368,3,1.000000,0.999878,24
10,full_ltn_qwen_context_interaction,test_true_selective_repeated_seed,bearish,0.9,2775,213,7.675676,255,9.189189,43.661972,0.087537,8.913219e-01,-0.087537,99,93,21,62,-2.202261,2.175598,0.5200,4,1.000000,0.999720,24


,seed,variant,n_candidate_rules,n_selected_rules,validation_n_min,validation_n_median,validation_n_max,validation_accuracy_mean
0,1,full_ltn_qwen_context_interaction,1536,24,12,44.0,126,92.981862
1,2,full_ltn_qwen_context_interaction,1536,24,12,44.0,126,92.981862
2,3,full_ltn_qwen_context_interaction,1536,24,12,44.0,126,92.981862
3,4,full_ltn_qwen_context_interaction,1536,24,12,44.0,126,92.981862
4,5,full_ltn_qwen_context_interaction,1536,24,12,44.0,126,92.981862
5,7,full_ltn_qwen_context_interaction,1536,24,12,44.0,126,92.981862
6,11,full_ltn_qwen_context_interaction,1536,24,12,44.0,126,92.981862
7,13,full_ltn_qwen_context_interaction,1536,24,12,44.0,126,92.981862
8,17,full_ltn_qwen_context_interaction,1536,24,12,44.0,126,92.981862
9,19,full_ltn_qwen_context_interaction,1536,24,12,44.0,126,92.981862


,variant,signal,n_seeds,n_total,n_signal_mean,n_signal_min,n_signal_max,coverage_mean,coverage_std,accuracy_mean,accuracy_std,accuracy_min,accuracy_max,aligned_mean_mean,aligned_mean_std,p_value_median,selected_rules_mean,selected_rules_min,selected_rules_max,selected_true_bullish_mean,selected_true_bearish_mean,selected_true_neutral_mean
0,full_ltn_qwen_context_interaction,bearish,10,2775,214.7,137,268,7.736937,1.440946,51.704091,5.403869,43.661972,62.773723,0.048524,0.613599,0.328790,24.0,24,24,91.8,110.0,12.9
1,full_ltn_qwen_context_interaction,bullish,10,2775,249.1,199,332,8.976577,1.563635,63.397568,4.696563,57.281553,70.854271,2.623942,0.369620,0.000004,24.0,24,24,157.9,73.4,17.8
2,full_ltn_qwen_context_interaction,joint,10,2775,463.8,369,545,16.713514,1.797858,57.883738,3.346354,50.652174,62.059621,1.405539,0.456329,0.000008,24.0,24,24,249.7,183.4,30.7


**Reading this table:** across the same 10 seeds, high-confidence (0.90 quantile) *bullish* selective predictions are significant in every single seed (median bootstrap p = 0.000004, aligned mean return 2.62% $\pm$ 0.37%, accuracy ranging 57.3%-70.9%). *Bearish* selective predictions are not (median p = 0.33, aligned mean return close to zero). This is worth flagging alongside the validation-only threshold grid earlier in this section, which showed the **opposite** pattern -- bearish aligned return improving with confidence, bullish staying negative throughout. That the test-set outcome inverts what validation predicted suggests this specific bullish/bearish asymmetry may be a feature of this particular test window (Mar-Jun 2026) rather than a stable, generalisable property of the model's confidence calibration. It is reported here as an observation on this test window, not as a general claim about the model.

In [23]:
# Inspect rules and successful selective predictions safely

if "true_test_selective" not in globals():
    raise NameError("Run the selective classifier cells first so true_test_selective exists.")

if "spec" not in globals() or "selected_rules" not in spec:
    raise NameError("Run the frozen-rule / selective cells first so spec['selected_rules'] exists.")

selected_rules_display = spec["selected_rules"].copy()

print("Selected rules columns:")
display(pd.DataFrame({"columns": selected_rules_display.columns}))

# Try to detect useful columns automatically
label_col_candidates = [
    "label", "target_label", "consequent", "class", "prediction", "outcome"
]
acc_col_candidates = [
    "validation_accuracy_pct", "accuracy_pct", "val_accuracy_pct",
    "validation_accuracy", "accuracy"
]

label_col = next(
    (c for c in label_col_candidates if c in selected_rules_display.columns),
    None
)

acc_col = next(
    (c for c in acc_col_candidates if c in selected_rules_display.columns),
    None
)

sort_cols = []
ascending = []

if label_col:
    sort_cols.append(label_col)
    ascending.append(True)

if acc_col:
    sort_cols.append(acc_col)
    ascending.append(False)

print("Detected label column:", label_col)
print("Detected accuracy column:", acc_col)

print("Selected LTN rules used by the selective model")
if sort_cols:
    display(
        selected_rules_display
        .sort_values(sort_cols, ascending=ascending)
        .reset_index(drop=True)
    )
else:
    display(selected_rules_display.reset_index(drop=True))

successful_selective = true_test_selective.loc[
    true_test_selective["has_signal"]
    & true_test_selective["selective_prediction"].eq(true_test_selective["target_label"])
].copy()

failed_selective = true_test_selective.loc[
    true_test_selective["has_signal"]
    & ~true_test_selective["selective_prediction"].eq(true_test_selective["target_label"])
].copy()

print("Selective prediction summary")
display(pd.DataFrame([{
    "total_test_events": len(true_test_selective),
    "selected_events": int(true_test_selective["has_signal"].sum()),
    "successful_selected_events": len(successful_selective),
    "failed_selected_events": len(failed_selective),
    "selective_accuracy_pct": 100 * len(successful_selective) / max(1, int(true_test_selective["has_signal"].sum())),
    "coverage_pct": 100 * int(true_test_selective["has_signal"].sum()) / max(1, len(true_test_selective)),
}]))

cols_to_show = [
    "event_date",
    "ticker",
    "title",
    "headline",
    "summary",
    "target_label",
    "prediction",
    "selective_prediction",
    "simple_return",
    "score_bullish",
    "score_bearish",
    "score_neutral",
    "confidence_margin",
]

cols_to_show = [c for c in cols_to_show if c in true_test_selective.columns]

print("Successful selective predictions")
display(
    successful_selective[cols_to_show]
    .sort_values("confidence_margin", ascending=False)
    .reset_index(drop=True)
)

print("Failed selective predictions")
display(
    failed_selective[cols_to_show]
    .sort_values("confidence_margin", ascending=False)
    .reset_index(drop=True)
)

# If a label-like column exists, break rules down by bullish/bearish
if label_col:
    for signal in ["bullish", "bearish"]:
        print(f"\nRules associated with {signal.upper()}")
        signal_rules = selected_rules_display.loc[
            selected_rules_display[label_col].astype(str).str.lower().eq(signal)
        ].copy()

        if len(signal_rules) == 0:
            print(f"No selected rules found for {signal}.")
        else:
            display(signal_rules.reset_index(drop=True))

        print(f"Successful {signal.upper()} selective events")
        display(
            successful_selective.loc[
                successful_selective["selective_prediction"].eq(signal),
                cols_to_show
            ]
            .sort_values("confidence_margin", ascending=False)
            .reset_index(drop=True)
        )
else:
    print("Could not find a label/class column in selected_rules, so rule breakdown by bullish/bearish was skipped.")

Selected rules columns:


,columns
0,condition
1,condition_cols_json
2,rule_label
3,rule_score
4,rule_blocks
5,training_n
6,training_accuracy_pct
7,training_aligned_mean_pct
8,training_aligned_signflip_p
9,validation_n


Detected label column: None
Detected accuracy column: validation_accuracy_pct
Selected LTN rules used by the selective model


,condition,condition_cols_json,rule_label,rule_score,rule_blocks,training_n,training_accuracy_pct,training_aligned_mean_pct,training_aligned_signflip_p,validation_n,validation_accuracy_pct,validation_aligned_mean_pct,validation_aligned_signflip_p
0,novel_context_30d & log1p_same_day_article_eve...,"[""novel_context_30d"", ""log1p_same_day_article_...",bullish,4.268611,"market_context, qwen_semantic",2776,69.344380,1.150335,6.603166e-109,86,100.000000,2.675768,1.292470e-26
1,log1p_same_day_article_event_count,"[""log1p_same_day_article_event_count""]",bullish,3.600519,qwen_semantic,3850,60.051948,-0.369321,4.647784e-47,86,100.000000,2.675768,1.292470e-26
2,has_article_event & log1p_same_day_article_eve...,"[""has_article_event"", ""log1p_same_day_article_...",bullish,3.600519,qwen_semantic,3850,60.051948,-0.369321,4.647784e-47,86,100.000000,2.675768,1.292470e-26
3,company_relevance_direct_company & log1p_same_...,"[""company_relevance_direct_company"", ""log1p_sa...",bullish,3.600519,qwen_semantic,3850,60.051948,-0.369321,4.647784e-47,86,100.000000,2.675768,1.292470e-26
4,confidence_high & log1p_same_day_article_event...,"[""confidence_high"", ""log1p_same_day_article_ev...",bullish,3.606061,qwen_semantic,3597,60.606061,-0.444858,2.280395e-47,74,100.000000,2.772048,5.293956e-23
5,log1p_same_day_article_event_count & ticker_pr...,"[""log1p_same_day_article_event_count"", ""ticker...",bullish,4.629755,"market_context, qwen_semantic",1447,81.340705,1.632696,5.779862e-152,61,100.000000,3.414603,4.336809e-19
6,log1p_same_day_article_event_count & relative_...,"[""log1p_same_day_article_event_count"", ""relati...",bullish,4.333374,"market_context, qwen_semantic",1112,78.507194,1.096604,2.356332e-100,61,100.000000,3.414603,4.336809e-19
7,interaction__direction_positive__x__novel_cont...,"[""interaction__direction_positive__x__novel_co...",bullish,4.335764,"qwen_context_interaction, qwen_semantic",1931,70.792336,1.255681,1.639087e-86,59,100.000000,2.862050,1.734723e-18
8,log1p_same_day_article_event_count & interacti...,"[""log1p_same_day_article_event_count"", ""intera...",bullish,4.406128,"qwen_context_interaction, qwen_semantic",993,80.060423,1.211048,2.602816e-94,46,100.000000,3.424170,1.421085e-14
9,log1p_same_day_article_event_count & interacti...,"[""log1p_same_day_article_event_count"", ""intera...",bullish,4.104416,"qwen_context_interaction, qwen_semantic",564,64.007092,0.928690,1.460283e-14,42,100.000000,2.614004,2.273737e-13


Selective prediction summary


...

Successful selective predictions


,event_date,ticker,target_label,prediction,selective_prediction,simple_return,score_bullish,score_bearish,score_neutral,confidence_margin
0,2026-05-22,TSLA,bullish,bullish,bullish,0.031041,1.000000,7.751881e-15,2.437642e-19,1.000000
1,2026-05-07,TSLA,bullish,bullish,bullish,0.092741,1.000000,4.339720e-11,1.189526e-12,1.000000
2,2026-03-27,AAPL,bullish,bullish,bullish,0.007028,1.000000,7.639534e-15,1.980152e-09,1.000000
3,2026-04-15,MSFT,bullish,bullish,bullish,0.063425,1.000000,1.495133e-20,1.419558e-15,1.000000
4,2026-04-27,AAPL,bullish,bullish,bullish,0.051833,1.000000,1.697662e-09,1.918687e-09,1.000000
...,...,...,...,...,...,...,...,...,...,...
256,2026-05-26,AAPL,bearish,bearish,bearish,-0.010499,0.000210,9.997897e-01,2.154379e-07,0.999580
257,2026-05-26,AAPL,bearish,bearish,bearish,-0.010499,0.000077,9.997223e-01,2.004906e-04,0.999522
258,2026-03-16,NVDA,bearish,bearish,bearish,-0.057228,0.000265,9.997217e-01,1.353273e-05,0.999457
259,2026-03-09,AMZN,bearish,bearish,bearish,-0.013210,0.000274,9.997184e-01,7.624159e-06,0.999444


Failed selective predictions


,event_date,ticker,target_label,prediction,selective_prediction,simple_return,score_bullish,score_bearish,score_neutral,confidence_margin
0,2026-03-05,AAPL,neutral,bullish,bullish,-0.000844,1.000000,1.983377e-12,2.275684e-08,1.000000
1,2026-03-05,AAPL,neutral,bullish,bullish,-0.000844,1.000000,4.838062e-13,7.599983e-09,1.000000
2,2026-03-05,AAPL,neutral,bullish,bullish,-0.000844,1.000000,4.179454e-13,2.972261e-08,1.000000
3,2026-03-05,AAPL,neutral,bullish,bullish,-0.000844,1.000000,2.249664e-12,1.305824e-08,1.000000
4,2026-03-05,AAPL,neutral,bullish,bullish,-0.000844,1.000000,1.278841e-12,1.086900e-08,1.000000
...,...,...,...,...,...,...,...,...,...,...
168,2026-03-30,META,bullish,bearish,bearish,0.067321,0.000224,9.997757e-01,3.817465e-12,0.999551
169,2026-04-13,MSFT,bullish,bearish,bearish,0.129189,0.000234,9.997495e-01,1.609966e-05,0.999515
170,2026-04-14,MSFT,bullish,bearish,bearish,0.075393,0.000252,9.997476e-01,1.735357e-07,0.999495
171,2026-04-23,MSFT,bullish,bearish,bearish,0.008699,0.000273,9.997259e-01,9.824887e-07,0.999453


Could not find a label/class column in selected_rules, so rule breakdown by bullish/bearish was skipped.


In [24]:

# Evaluate selected rules on the held-out test set

import json
import pandas as pd
import numpy as np
from scipy.stats import binomtest

if "true_test_scored" not in globals():
    raise NameError("Run the selective final refit cell first so true_test_scored exists.")

if "spec" not in globals() or "selected_rules" not in spec:
    raise NameError("Run the selective/frozen-rule cells first so spec['selected_rules'] exists.")

rules = spec["selected_rules"].copy()
test_eval = true_test_scored.copy()

rows = []

for idx, rule in rules.iterrows():
    rule_label = rule.get("rule_label", rule.get("label", None))
    condition = rule.get("condition", "")

    # condition_cols_json usually stores the feature columns used by the rule
    raw_cols = rule.get("condition_cols_json", None)

    if isinstance(raw_cols, str):
        try:
            condition_cols = json.loads(raw_cols)
        except Exception:
            condition_cols = []
    elif isinstance(raw_cols, list):
        condition_cols = raw_cols
    else:
        condition_cols = []

    missing_cols = [c for c in condition_cols if c not in test_eval.columns]

    if missing_cols or not condition_cols:
        rows.append({
            "condition": condition,
            "rule_label": rule_label,
            "test_n": 0,
            "test_accuracy_pct": np.nan,
            "test_aligned_mean_pct": np.nan,
            "test_aligned_signflip_p": np.nan,
            "missing_cols": missing_cols,
        })
        continue

    # A row matches the rule if all condition columns are active/positive.
    # threshold at > 0.5
    antecedent_threshold = globals().get("ANTECEDENT_THRESHOLD", 0.50)

    # A row matches the rule if all antecedent predicates meet the rule-mining threshold.
    mask = np.ones(len(test_eval), dtype=bool)
    for col in condition_cols:
        mask &= (
            pd.to_numeric(test_eval[col], errors="coerce")
            .fillna(0)
            .ge(antecedent_threshold)
        )

    subset = test_eval.loc[mask].copy()

    if subset.empty:
        rows.append({
            "condition": condition,
            "rule_label": rule_label,
            "test_n": 0,
            "test_accuracy_pct": np.nan,
            "test_aligned_mean_pct": np.nan,
            "test_aligned_signflip_p": np.nan,
            "missing_cols": [],
        })
        continue

    test_accuracy = 100 * subset["target_label"].eq(rule_label).mean()

    # Directional aligned return for bullish/bearish rules
    if rule_label in ["bullish", "bearish"]:
        ret = pd.to_numeric(subset["simple_return"], errors="coerce")
        signed = ret if rule_label == "bullish" else -ret
        signed = signed.dropna()

        if len(signed) >= 3:
            aligned_mean = 100 * signed.mean()
            p_value = binomtest(
                int((signed > 0).sum()),
                len(signed),
                0.5,
                alternative="greater"
            ).pvalue
        else:
            aligned_mean = np.nan
            p_value = np.nan
    else:
        aligned_mean = np.nan
        p_value = np.nan

    rows.append({
        "condition": condition,
        "rule_label": rule_label,
        "training_n": rule.get("training_n", np.nan),
        "training_accuracy_pct": rule.get("training_accuracy_pct", np.nan),
        "validation_n": rule.get("validation_n", np.nan),
        "validation_accuracy_pct": rule.get("validation_accuracy_pct", np.nan),
        "test_n": int(len(subset)),
        "test_accuracy_pct": test_accuracy,
        "test_aligned_mean_pct": aligned_mean,
        "test_aligned_signflip_p": p_value,
        "missing_cols": [],
    })

rule_test_eval = pd.DataFrame(rows)

display(
    rule_test_eval
    .sort_values(["rule_label", "test_accuracy_pct", "test_n"], ascending=[True, False, False])
    .reset_index(drop=True)
)

,condition,rule_label,training_n,training_accuracy_pct,validation_n,validation_accuracy_pct,test_n,test_accuracy_pct,test_aligned_mean_pct,test_aligned_signflip_p,missing_cols
0,market_prior_down_5d & relative_prior_up_5d,bearish,340,65.294118,27,100.000000,155,67.741935,0.932751,2.235818e-07,[]
1,market_prior_down_5d & interaction__direction_...,bearish,273,73.626374,16,100.000000,108,67.592593,0.975441,1.380393e-05,[]
2,interaction__direction_positive__x__relative_p...,bearish,41,60.975610,12,91.666667,15,60.000000,1.333357,3.036194e-01,[]
3,interaction__direction_positive__x__relative_p...,bearish,38,60.526316,12,91.666667,15,60.000000,1.333357,3.036194e-01,[]
4,interaction__event_type_regulatory_legal_outco...,bearish,223,66.367713,37,89.189189,72,33.333333,-1.467984,9.936127e-01,[]
5,relative_prior_up_5d & event_type_regulatory_l...,bearish,223,66.367713,37,89.189189,72,33.333333,-1.467984,9.936127e-01,[]
6,relative_prior_up_5d & interaction__event_type...,bearish,220,66.363636,37,89.189189,72,33.333333,-1.467984,9.936127e-01,[]
7,interaction__event_type_earnings_guidance__x__...,bearish,642,77.258567,126,76.190476,278,26.978417,-1.957570,1.000000e+00,[]
8,event_type_earnings_guidance & relative_prior_...,bearish,642,77.258567,126,76.190476,278,26.978417,-1.957570,1.000000e+00,[]
9,event_type_earnings_guidance & interaction__di...,bearish,564,78.191489,99,76.767677,236,25.000000,-2.086009,1.000000e+00,[]


In [25]:
# Full + selective classifier diagnostics:
# accuracy, majority baseline, confusion matrix, precision/recall/F1,
# prediction distribution, and aligned return.

import numpy as np
import pandas as pd
from scipy.stats import binomtest

DIAGNOSTIC_LABELS = list(globals().get("LABELS", ["bearish", "bullish", "neutral"]))


def aligned_return_stats(df, prediction_col, return_col="simple_return"):
    ret = pd.to_numeric(df[return_col], errors="coerce")
    pred = df[prediction_col].astype(str)

    signed = np.select(
        [pred.eq("bullish"), pred.eq("bearish")],
        [ret, -ret],
        default=np.nan,
    )

    signed = pd.Series(signed, index=df.index).dropna()

    if len(signed) >= 3:
        aligned_mean_pct = float(100 * signed.mean())
        aligned_signflip_p = binomtest(
            int((signed > 0).sum()),
            len(signed),
            0.5,
            alternative="greater",
        ).pvalue
    else:
        aligned_mean_pct = np.nan
        aligned_signflip_p = np.nan

    neutral_mask = pred.eq("neutral")
    neutral_abs_return = ret.loc[neutral_mask].abs().dropna()

    return {
        "directional_prediction_n": int(len(signed)),
        "aligned_mean_pct": aligned_mean_pct,
        "aligned_signflip_p": float(aligned_signflip_p) if pd.notna(aligned_signflip_p) else np.nan,
        "neutral_prediction_n": int(neutral_mask.sum()),
        "neutral_mean_abs_return_pct": (
            float(100 * neutral_abs_return.mean()) if len(neutral_abs_return) else np.nan
        ),
    }


def classifier_diagnostics(
    df,
    diagnostic_name,
    prediction_col,
    target_col="target_label",
    return_col="simple_return",
    labels=DIAGNOSTIC_LABELS,
):
    work = df.copy()

    missing = [
        col for col in [prediction_col, target_col, return_col]
        if col not in work.columns
    ]
    if missing:
        raise ValueError(f"{diagnostic_name} missing columns: {missing}")

    work = work.loc[
        work[prediction_col].notna()
        & work[target_col].notna()
        & work[return_col].notna()
    ].copy()

    if work.empty:
        raise ValueError(f"{diagnostic_name} has no valid rows.")

    y_true = work[target_col].astype(str)
    y_pred = work[prediction_col].astype(str)

    accuracy_pct = float(100 * y_pred.eq(y_true).mean())
    majority_label = y_true.value_counts().idxmax()
    majority_baseline_pct = float(100 * y_true.value_counts(normalize=True).max())

    summary = {
        "diagnostic": diagnostic_name,
        "n": int(len(work)),
        "accuracy_pct": accuracy_pct,
        "majority_label": majority_label,
        "majority_baseline_pct": majority_baseline_pct,
        "accuracy_minus_majority_pct": accuracy_pct - majority_baseline_pct,
        "mean_simple_return_pct": float(100 * pd.to_numeric(work[return_col], errors="coerce").mean()),
    }

    summary.update(
        aligned_return_stats(
            work,
            prediction_col=prediction_col,
            return_col=return_col,
        )
    )

    summary = pd.DataFrame([summary])

    confusion = pd.crosstab(
        y_true,
        y_pred,
        rownames=["actual"],
        colnames=["predicted"],
        dropna=False,
    ).reindex(index=labels, columns=labels, fill_value=0)

    per_class_rows = []

    for label in labels:
        tp = int(((y_true == label) & (y_pred == label)).sum())
        fp = int(((y_true != label) & (y_pred == label)).sum())
        fn = int(((y_true == label) & (y_pred != label)).sum())
        tn = int(((y_true != label) & (y_pred != label)).sum())

        precision = tp / (tp + fp) if (tp + fp) else np.nan
        recall = tp / (tp + fn) if (tp + fn) else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if pd.notna(precision) and pd.notna(recall) and (precision + recall)
            else np.nan
        )

        per_class_rows.append({
            "diagnostic": diagnostic_name,
            "class": label,
            "support": int((y_true == label).sum()),
            "predicted_n": int((y_pred == label).sum()),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "precision_pct": float(100 * precision) if pd.notna(precision) else np.nan,
            "recall_pct": float(100 * recall) if pd.notna(recall) else np.nan,
            "f1_pct": float(100 * f1) if pd.notna(f1) else np.nan,
        })

    per_class = pd.DataFrame(per_class_rows)

    actual_distribution = (
        y_true.value_counts()
        .reindex(labels, fill_value=0)
        .rename_axis("class")
        .reset_index(name="actual_n")
    )
    actual_distribution["actual_pct"] = (
        100 * actual_distribution["actual_n"] / len(work)
    )

    prediction_distribution = (
        y_pred.value_counts()
        .reindex(labels, fill_value=0)
        .rename_axis("class")
        .reset_index(name="predicted_n")
    )
    prediction_distribution["predicted_pct"] = (
        100 * prediction_distribution["predicted_n"] / len(work)
    )

    distribution = actual_distribution.merge(
        prediction_distribution,
        on="class",
        how="outer",
    )
    distribution["diagnostic"] = diagnostic_name

    return summary, confusion, per_class, distribution


diagnostic_tables = {}

# FULL CLASSIFIER
# true_test_scored is produced by the final selective refit cell.
# It contains the full classifier prediction for every test row.
if "true_test_scored" not in globals():
    raise NameError(
        "Run the final test scoring cell first. Expected dataframe: true_test_scored"
    )

full_summary, full_confusion, full_per_class, full_distribution = classifier_diagnostics(
    true_test_scored,
    diagnostic_name="full_classifier_test",
    prediction_col="prediction",
)

diagnostic_tables["full_classifier_summary"] = full_summary
diagnostic_tables["full_classifier_confusion"] = full_confusion
diagnostic_tables["full_classifier_per_class"] = full_per_class
diagnostic_tables["full_classifier_distribution"] = full_distribution

print("FULL CLASSIFIER SUMMARY")
display(full_summary)

print("FULL CLASSIFIER CONFUSION MATRIX")
display(full_confusion)

print("FULL CLASSIFIER PER-CLASS PRECISION / RECALL / F1")
display(full_per_class)

print("FULL CLASSIFIER CLASS DISTRIBUTION")
display(full_distribution)


# SELECTIVE CLASSIFIER
# true_test_selective is produced by the selective threshold cell.
# It contains selective_prediction and has_signal.
if "true_test_selective" not in globals():
    raise NameError(
        "Run the selective test cell first. Expected dataframe: true_test_selective"
    )

selective_test = true_test_selective.loc[
    true_test_selective["has_signal"].astype(bool)
].copy()

selective_summary, selective_confusion, selective_per_class, selective_distribution = classifier_diagnostics(
    selective_test,
    diagnostic_name="selective_classifier_test",
    prediction_col="selective_prediction",
)

selective_summary["selected_n"] = len(selective_test)
selective_summary["total_test_n"] = len(true_test_selective)
selective_summary["coverage_pct"] = (
    100 * len(selective_test) / len(true_test_selective)
)

diagnostic_tables["selective_classifier_summary"] = selective_summary
diagnostic_tables["selective_classifier_confusion"] = selective_confusion
diagnostic_tables["selective_classifier_per_class"] = selective_per_class
diagnostic_tables["selective_classifier_distribution"] = selective_distribution

print("SELECTIVE CLASSIFIER SUMMARY")
display(selective_summary)

print("SELECTIVE CLASSIFIER CONFUSION MATRIX")
display(selective_confusion)

print("SELECTIVE CLASSIFIER PER-CLASS PRECISION / RECALL / F1")
display(selective_per_class)

print("SELECTIVE CLASSIFIER CLASS DISTRIBUTION")
display(selective_distribution)


# Combined summary tables for easier dissertation reporting
combined_summary = pd.concat(
    [full_summary, selective_summary],
    ignore_index=True,
)

combined_per_class = pd.concat(
    [full_per_class, selective_per_class],
    ignore_index=True,
)

combined_distribution = pd.concat(
    [full_distribution, selective_distribution],
    ignore_index=True,
)

print("COMBINED SUMMARY")
display(combined_summary)

print("COMBINED PER-CLASS METRICS")
display(combined_per_class)

print("COMBINED CLASS DISTRIBUTION")
display(combined_distribution)


# Save outputs if your notebook is already saving results
if "SAVE_OUTPUTS" in globals() and SAVE_OUTPUTS and "OUTPUT_DIR" in globals():
    combined_summary.to_csv(
        OUTPUT_DIR / "full_and_selective_classifier_summary.csv",
        index=False,
    )
    combined_per_class.to_csv(
        OUTPUT_DIR / "full_and_selective_classifier_per_class_metrics.csv",
        index=False,
    )
    combined_distribution.to_csv(
        OUTPUT_DIR / "full_and_selective_classifier_class_distribution.csv",
        index=False,
    )
    full_confusion.to_csv(
        OUTPUT_DIR / "full_classifier_confusion_matrix.csv",
    )
    selective_confusion.to_csv(
        OUTPUT_DIR / "selective_classifier_confusion_matrix.csv",
    )

    print("Saved full and selective classifier diagnostics to:", OUTPUT_DIR)

FULL CLASSIFIER SUMMARY


,diagnostic,n,accuracy_pct,majority_label,majority_baseline_pct,accuracy_minus_majority_pct,mean_simple_return_pct,directional_prediction_n,aligned_mean_pct,aligned_signflip_p,neutral_prediction_n,neutral_mean_abs_return_pct
0,full_classifier_test,2775,42.45045,bullish,50.882883,-8.432432,1.09779,2563,0.213052,0.807607,212,3.459573


FULL CLASSIFIER CONFUSION MATRIX


predicted,bearish,bullish,neutral
actual,,,
bearish,349,667,92
bullish,507,807,98
neutral,85,148,22


FULL CLASSIFIER PER-CLASS PRECISION / RECALL / F1


...

COMBINED PER-CLASS METRICS


,diagnostic,class,support,predicted_n,tp,fp,fn,tn,precision_pct,recall_pct,f1_pct
0,full_classifier_test,bearish,1108,941,349,592,759,1075,37.088204,31.498195,34.065398
1,full_classifier_test,bullish,1412,1622,807,815,605,548,49.753391,57.152975,53.197100
2,full_classifier_test,neutral,255,212,22,190,233,2330,10.377358,8.627451,9.421842
3,selective_classifier_test,bearish,178,223,122,101,56,155,54.708520,68.539326,60.847880
4,selective_classifier_test,bullish,229,211,139,72,90,133,65.876777,60.698690,63.181818
5,selective_classifier_test,neutral,27,0,0,0,27,407,NaN,0.000000,NaN


COMBINED CLASS DISTRIBUTION


,class,actual_n,actual_pct,predicted_n,predicted_pct,diagnostic
0,bearish,1108,39.927928,941,33.909910,full_classifier_test
1,bullish,1412,50.882883,1622,58.450450,full_classifier_test
2,neutral,255,9.189189,212,7.639640,full_classifier_test
3,bearish,178,41.013825,223,51.382488,selective_classifier_test
4,bullish,229,52.764977,211,48.617512,selective_classifier_test
5,neutral,27,6.221198,0,0.000000,selective_classifier_test


Saved full and selective classifier diagnostics to: /home/jovyan/Stock-Sentiment-Prediction/FINAL_17_mag7_article_event_label_interaction_clean_no_robust


## 13. Longer-Window Robustness Test

This section repeats the frozen-rule evaluation from Section 9 on a second split, to test whether the primary-split results are stable under a different held-out window. An earlier attempt to shift the primary split's boundaries backward in time produced a severely underpowered validation set (266 rows against a 10,377-row test set) -- a symptom of a sharp, likely collection-driven increase in article volume around October 2025 rather than a genuine trend (see Section 16 for discussion). This split instead stays entirely within that higher-volume period (training: 2025-10-01 to 2026-02-28; validation: 2026-03-01 to 2026-03-31; test: 2026-04-01 to 2026-06-01), which avoids straddling that discontinuity at the cost of testing robustness within one era rather than across two. Note that this test window (2 months) is smaller than the primary split's test window (3 months); it is a differently-positioned window, not a larger one.

In [26]:
def build_feature_frame_for_windows(training, validation, test):
    """
    Parameterised rebuild of the Section 4/5 feature pipeline for an
    arbitrary (training, validation, test) window. Reuses the already-loaded
    `ctx` price/context/event columns and the module-level scaling helper
    functions (soft_pos_with_scale, soft_neg_with_scale, soft_low_with_cutoff,
    soft_high_with_cutoff), but refits all training-only scales, dummy
    categories, and interaction features on the new window's own training
    period rather than the primary split's.
    """

    def assign_period_r(d):
        d = pd.Timestamp(d)
        if pd.Timestamp(training[0]) <= d <= pd.Timestamp(training[1]):
            return "training"
        if pd.Timestamp(validation[0]) <= d <= pd.Timestamp(validation[1]):
            return "validation"
        if pd.Timestamp(test[0]) <= d <= pd.Timestamp(test[1]):
            return "test"
        return "outside"

    ctx_r = ctx.drop(columns=["period"], errors="ignore").copy()
    ctx_r["period"] = ctx_r["event_date"].map(assign_period_r)
    fit_mask_r = ctx_r["period"].eq("training")

    def fit_abs_scale_r(col, q=0.80):
        s = pd.to_numeric(ctx_r.loc[fit_mask_r, col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        scale = s.abs().quantile(q)
        return float(scale) if np.isfinite(scale) and scale != 0 else 1.0

    def fit_quantile_r(col, q, fallback=1.0):
        s = pd.to_numeric(ctx_r.loc[fit_mask_r, col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        value = s.quantile(q)
        return float(value) if np.isfinite(value) and value != 0 else fallback

    ticker_5d_scale_r = fit_abs_scale_r("ticker_ret_5d_pre")
    market_5d_scale_r = fit_abs_scale_r("sp500_ret_5d_pre")
    relative_5d_scale_r = fit_abs_scale_r("relative_ret_5d_pre")

    ctx_r["ticker_prior_up_5d"] = soft_pos_with_scale(ctx_r["ticker_ret_5d_pre"], ticker_5d_scale_r)
    ctx_r["ticker_prior_down_5d"] = soft_neg_with_scale(ctx_r["ticker_ret_5d_pre"], ticker_5d_scale_r)
    ctx_r["market_prior_up_5d"] = soft_pos_with_scale(ctx_r["sp500_ret_5d_pre"], market_5d_scale_r)
    ctx_r["market_prior_down_5d"] = soft_neg_with_scale(ctx_r["sp500_ret_5d_pre"], market_5d_scale_r)
    ctx_r["relative_prior_up_5d"] = soft_pos_with_scale(ctx_r["relative_ret_5d_pre"], relative_5d_scale_r)
    ctx_r["relative_prior_down_5d"] = soft_neg_with_scale(ctx_r["relative_ret_5d_pre"], relative_5d_scale_r)
    ctx_r["quiet_market_5d"] = soft_low_with_cutoff(ctx_r["sp500_vol_5d_pre"], fit_quantile_r("sp500_vol_5d_pre", 0.40))
    ctx_r["volatile_market_5d"] = soft_high_with_cutoff(ctx_r["sp500_vol_5d_pre"], fit_quantile_r("sp500_vol_5d_pre", 0.70))
    ctx_r["quiet_ticker_5d"] = soft_low_with_cutoff(ctx_r["ticker_vol_5d_pre"], fit_quantile_r("ticker_vol_5d_pre", 0.40))
    ctx_r["volatile_ticker_5d"] = soft_high_with_cutoff(ctx_r["ticker_vol_5d_pre"], fit_quantile_r("ticker_vol_5d_pre", 0.70))

    for col in ["event_density_ticker_7d", "event_density_ticker_30d", "similar_density_ticker_30d"]:
        denom = max(ctx_r.loc[fit_mask_r, col].quantile(0.95), 1)
        ctx_r[col + "_01"] = (ctx_r[col] / denom).clip(0, 1)

    ctx_r["novel_context_30d"] = (1 - ctx_r["similar_density_ticker_30d_01"]).clip(0, 1)

    dummy_parts_r = []
    dummy_cols_r = []
    categorical_cols_r = ["event_type", "direction", "confidence", "company_relevance"]

    for col in categorical_cols_r:
        categories = sorted(set(ctx_r.loc[fit_mask_r, col].fillna("missing").astype(str)))
        values = ctx_r[col].fillna("missing").astype(str)
        for category in categories:
            dummy_name = f"{col}_{category}"
            dummy_parts_r.append(values.eq(category).astype(float).rename(dummy_name))
            dummy_cols_r.append(dummy_name)

    dummies_r = pd.concat(dummy_parts_r, axis=1) if dummy_parts_r else pd.DataFrame(index=ctx_r.index)
    ctx_r = ctx_r.drop(columns=dummy_cols_r, errors="ignore")
    ctx_r = pd.concat([ctx_r, dummies_r], axis=1)

    count_feature_cols_r = []
    for col in ["same_day_article_event_count", "same_day_event_type_direction_count"]:
        feature_col = f"log1p_{col}"
        ctx_r[feature_col] = np.log1p(pd.to_numeric(ctx_r[col], errors="coerce").fillna(0))
        denom = max(ctx_r.loc[fit_mask_r, feature_col].quantile(0.95), 1e-9)
        ctx_r[feature_col] = (ctx_r[feature_col] / denom).clip(0, 1)
        count_feature_cols_r.append(feature_col)

    ctx_r["article_event_weight_01"] = pd.to_numeric(ctx_r["article_event_weight"], errors="coerce").fillna(0).clip(0, 1)

    context_cols_r = [
        "ticker_prior_up_5d", "ticker_prior_down_5d", "market_prior_up_5d", "market_prior_down_5d",
        "relative_prior_up_5d", "relative_prior_down_5d", "quiet_market_5d", "volatile_market_5d",
        "quiet_ticker_5d", "volatile_ticker_5d",
        "event_density_ticker_7d_01", "event_density_ticker_30d_01", "similar_density_ticker_30d_01",
        "novel_context_30d", "hour_sin_01", "hour_cos_01", "is_monday", "is_friday",
    ]

    qwen_cols_r = list(dummies_r.columns) + count_feature_cols_r + ["article_event_weight_01", "has_article_event"]

    interaction_qwen_cols_r = [
        c for c in dummy_cols_r
        if c.startswith("event_type_")
        or c.startswith("direction_")
    ]

    interaction_context_cols_r = [
        "relative_prior_up_5d", "relative_prior_down_5d", "quiet_market_5d", "volatile_market_5d",
        "quiet_ticker_5d", "volatile_ticker_5d", "event_density_ticker_7d_01", "novel_context_30d",
    ]

    interaction_data_r = {}
    for q in interaction_qwen_cols_r:
        for c in interaction_context_cols_r:
            name = f"interaction__{q}__x__{c}"
            interaction_data_r[name] = ctx_r[q].fillna(0) * ctx_r[c].fillna(0)

    interaction_cols_r = list(interaction_data_r.keys())
    ctx_r = ctx_r.drop(columns=interaction_cols_r, errors="ignore")
    if interaction_data_r:
        ctx_r = pd.concat([ctx_r, pd.DataFrame(interaction_data_r, index=ctx_r.index)], axis=1)

    feature_df_r = ctx_r.loc[ctx_r["period"].isin(["training", "validation", "test"])].copy()

    for col in qwen_cols_r + context_cols_r + interaction_cols_r:
        feature_df_r[col] = pd.to_numeric(feature_df_r[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 1)

    variant_specs_r = {
        "full_ltn_qwen_only": {"feature_cols": qwen_cols_r, "rule_blocks": ["qwen_semantic"]},
        "full_ltn_context_only": {"feature_cols": context_cols_r, "rule_blocks": ["market_context"]},
        "full_ltn_qwen_plus_context": {"feature_cols": qwen_cols_r + context_cols_r, "rule_blocks": ["qwen_semantic", "market_context"]},
        "full_ltn_qwen_context_interaction": {"feature_cols": qwen_cols_r + context_cols_r + interaction_cols_r, "rule_blocks": ["qwen_semantic", "market_context", "qwen_context_interaction"]},
    }

    for spec in variant_specs_r.values():
        spec["feature_cols"] = list(dict.fromkeys(spec["feature_cols"]))
        spec["return_col"] = "simple_return"

    return feature_df_r, variant_specs_r


def build_model_df_for_horizon_from_feature_frame(feature_frame, horizon: int) -> pd.DataFrame:
    horizon_returns = simple_returns.loc[
        simple_returns["horizon_days"].eq(int(horizon)),
        ["ticker", "event_date", "simple_return", "event_open", "event_close"],
    ]

    out = (
        feature_frame
        .drop(
            columns=["simple_return", "prev_close", "future_close", "event_open", "event_close", "target_label"],
            errors="ignore",
        )
        .merge(horizon_returns, on=["ticker", "event_date"], how="left")
    ).copy()

    missing_returns = int(out["simple_return"].isna().sum())
    if missing_returns:
        print(f"robust {horizon}d horizon: dropping {missing_returns} rows with missing open-to-close returns")
        out = out.loc[out["simple_return"].notna()].copy()

    out["target_label"] = out["simple_return"].map(
        lambda x: realised_label_from_return(float(x), RETURN_THRESHOLD)
    )

    return out


ROBUST_TRAINING = ("2025-10-01", "2026-02-28")
ROBUST_VALIDATION = ("2026-03-01", "2026-03-31")
ROBUST_TEST = ("2026-04-01", "2026-06-01")

robust_feature_df, robust_variant_specs = build_feature_frame_for_windows(
    ROBUST_TRAINING,
    ROBUST_VALIDATION,
    ROBUST_TEST,
)

model_df_robust = build_model_df_for_horizon_from_feature_frame(
    robust_feature_df,
    PRIMARY_HORIZON,
)

print("Robust split window:")
print("  training:", ROBUST_TRAINING)
print("  validation:", ROBUST_VALIDATION)
print("  test:", ROBUST_TEST)

display(model_df_robust.groupby("period")["target_label"].value_counts().unstack(fill_value=0))

robust_feature_manifest = pd.DataFrame([
    {
        "variant": name,
        "n_features": len(spec["feature_cols"]),
        "rule_blocks": ", ".join(spec["rule_blocks"]),
    }
    for name, spec in robust_variant_specs.items()
])
display(robust_feature_manifest)

train_val_robust = model_df_robust.loc[model_df_robust["period"].isin(["training", "validation"])].copy()
test_robust = model_df_robust.loc[model_df_robust["period"].eq("test")].copy()

frozen_robust_variant_specs = {}

for name, spec in robust_variant_specs.items():
    print("selecting robust frozen rules", name)

    candidate_rules, selected_rules = mine_validated_rules(
        train_val_robust,
        spec["feature_cols"],
        spec["return_col"],
        rule_blocks=spec.get("rule_blocks"),
        max_pair_features=spec.get("max_pair_features", 36),
        max_rules_per_label=spec.get("max_rules_per_label", 12),
    )

    frozen_spec = spec.copy()
    frozen_spec["selected_rules"] = selected_rules.copy()
    frozen_robust_variant_specs[name] = frozen_spec

trained_robust = {}

for name, spec in frozen_robust_variant_specs.items():
    print("robust frozen-rule training", name, "rows", len(train_val_robust))

    trained_robust[name] = train_variant(
        name,
        spec,
        train_val_robust,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=SEED,
        mine_rules=False,
    )

robust_rows = []
robust_frames = []

for variant, bundle in trained_robust.items():
    scored = score_variant(bundle, test_robust)

    robust_rows.append(
        summary_row(
            variant,
            "robust_test_frozen_rules_refit_training_validation",
            scored,
            bundle=bundle,
        )
    )

    robust_frames.append(
        scored.assign(
            variant=variant,
            period="robust_test_frozen_rules_refit_training_validation",
        )
    )

robust_final_summary = pd.DataFrame(robust_rows)

if SAVE_OUTPUTS:
    robust_final_summary.to_csv(
        OUTPUT_DIR / "robust_full_ltn_final_frozen_rule_test_summary.csv",
        index=False,
    )
    pd.concat(robust_frames, ignore_index=True).to_csv(
        OUTPUT_DIR / "robust_full_ltn_final_frozen_rule_test_predictions.csv",
        index=False,
    )

display(
    robust_final_summary.sort_values(
        ["accuracy_pct", "aligned_mean_pct"],
        ascending=False,
    )
)


robust 5d horizon: dropping 182 rows with missing open-to-close returns
Robust split window:
  training: ('2025-10-01', '2026-02-28')
  validation: ('2026-03-01', '2026-03-31')
  test: ('2026-04-01', '2026-06-01')


target_label,bearish,bullish,neutral
period,,,
test,395,1095,143
training,3206,3940,456
validation,713,317,112


,variant,n_features,rule_blocks
0,full_ltn_qwen_only,22,qwen_semantic
1,full_ltn_context_only,18,market_context
2,full_ltn_qwen_plus_context,40,"qwen_semantic, market_context"
3,full_ltn_qwen_context_interaction,160,"qwen_semantic, market_context, qwen_context_in..."


selecting robust frozen rules full_ltn_qwen_only
selecting robust frozen rules full_ltn_context_only
selecting robust frozen rules full_ltn_qwen_plus_context
selecting robust frozen rules full_ltn_qwen_context_interaction
robust frozen-rule training full_ltn_qwen_only rows 8744
robust frozen-rule training full_ltn_context_only rows 8744
robust frozen-rule training full_ltn_qwen_plus_context rows 8744
robust frozen-rule training full_ltn_qwen_context_interaction rows 8744


,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,mean_max_class_score,n_independent_clusters,aligned_mean_ci_low_pct,aligned_mean_ci_high_pct,aligned_bootstrap_p_value,use_logic,logic_weight,n_selected_rules
3,full_ltn_qwen_context_interaction,robust_test_frozen_rules_refit_training_valida...,1633,47.826087,2.72948,0.758264,0.001020,47.091243,45.682792,7.225964,0.866234,0.929177,204,-0.045075,1.587376,0.0328,True,0.3,15
2,full_ltn_qwen_plus_context,robust_test_frozen_rules_refit_training_valida...,1633,44.886712,2.72948,0.302136,0.142615,44.396816,45.744029,9.859155,0.822195,0.906572,206,-0.498595,1.123252,0.2428,True,0.3,20
1,full_ltn_context_only,robust_test_frozen_rules_refit_training_valida...,1633,42.498469,2.72948,-0.098639,0.759200,43.723209,53.214942,3.061849,0.663972,0.813155,207,-0.976079,0.791436,0.5918,True,0.3,13
0,full_ltn_qwen_only,robust_test_frozen_rules_refit_training_valida...,1633,33.251684,2.72948,-1.433674,1.000000,21.677893,78.199633,0.122474,0.240869,0.582233,207,-1.919807,-0.961144,1.0000,True,0.3,3


**Reading this table:** on this window, the Section 9 ranking reverses. `full_ltn_qwen_context_interaction` is now the most accurate variant (47.8%) and the only significant one (p = 0.033), while `full_ltn_context_only` -- the strongest, only-significant performer on the primary test -- drops to a negative, non-significant aligned return (-0.10%, p = 0.76). Section 16 discusses what this reversal implies.

## 14. Longer-Window Single-Seed LTN Versus No-LTN Ablation

This repeats the Section 10 LTN-versus-no-logic comparison on the longer-window robustness split, using the same seed as the primary ablation.

In [27]:
robust_ablation_base_spec = robust_variant_specs[PRIMARY_VARIANT]

robust_candidate_rules, robust_selected_rules = mine_validated_rules(
    train_val_robust,
    robust_ablation_base_spec["feature_cols"],
    robust_ablation_base_spec["return_col"],
    rule_blocks=robust_ablation_base_spec.get("rule_blocks"),
    max_pair_features=robust_ablation_base_spec.get("max_pair_features", 36),
    max_rules_per_label=robust_ablation_base_spec.get("max_rules_per_label", 12),
)

robust_ablation_ltn_spec = robust_ablation_base_spec.copy()
robust_ablation_ltn_spec["selected_rules"] = robust_selected_rules.copy()

robust_ltn_bundle = train_variant(
    "robust_ltn_qwen_context_interaction_ablation",
    robust_ablation_ltn_spec,
    train_val_robust,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

robust_ltn_scored = score_variant(robust_ltn_bundle, test_robust)

robust_no_logic_bundle = train_variant(
    "robust_no_logic_qwen_context_interaction_ablation",
    robust_ablation_base_spec,
    train_val_robust,
    use_logic=False,
    logic_weight=0.0,
    seed=SEED,
    mine_rules=False,
)

robust_no_logic_scored = score_variant(robust_no_logic_bundle, test_robust)

robust_ltn_ablation_summary = pd.DataFrame([
    summary_row(
        "LTN, article-event + context interaction",
        "robust_heldout_test",
        robust_ltn_scored,
        bundle=robust_ltn_bundle,
    ),
    summary_row(
        "Neural classifier, Qwen + context interaction, no logic",
        "robust_heldout_test",
        robust_no_logic_scored,
        bundle=robust_no_logic_bundle,
    ),
])

display(robust_ltn_ablation_summary)

if SAVE_OUTPUTS:
    robust_ltn_bundle["history"].to_csv(
        OUTPUT_DIR / "robust_ltn_qwen_context_interaction_ablation_training_history.csv",
        index=False,
    )
    robust_no_logic_bundle["history"].to_csv(
        OUTPUT_DIR / "robust_no_logic_qwen_context_interaction_ablation_training_history.csv",
        index=False,
    )
    robust_ltn_scored.to_csv(
        OUTPUT_DIR / "robust_ltn_qwen_context_interaction_ablation_test_predictions.csv",
        index=False,
    )
    robust_no_logic_scored.to_csv(
        OUTPUT_DIR / "robust_no_logic_qwen_context_interaction_ablation_test_predictions.csv",
        index=False,
    )
    robust_ltn_ablation_summary.to_csv(
        OUTPUT_DIR / "robust_ltn_ablation_qwen_context_interaction_summary.csv",
        index=False,
    )


,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,mean_max_class_score,n_independent_clusters,aligned_mean_ci_low_pct,aligned_mean_ci_high_pct,aligned_bootstrap_p_value,use_logic,logic_weight,n_selected_rules
0,"LTN, article-event + context interaction",robust_heldout_test,1633,47.826087,2.72948,0.744524,0.001471,47.703613,45.560318,6.736069,0.860515,0.925889,206,-0.013014,1.555594,0.0274,True,0.3,15
1,"Neural classifier, Qwen + context interaction,...",robust_heldout_test,1633,42.865891,2.72948,0.110781,0.815491,45.682792,47.274954,7.042254,0.839373,0.915224,206,-0.618013,0.889630,0.4078,False,0.0,0


## 15. Longer-Window Repeated-Seed LTN Versus No-LTN Ablation

This repeats the Section 11 repeated-seed comparison on the longer-window robustness split, across the same ten seeds, to check whether the LTN-versus-no-logic result is stable under a different held-out test window.

In [28]:
robust_seed_rows = []
robust_seed_rule_rows = []

robust_base_spec = robust_variant_specs[PRIMARY_VARIANT]

for seed in ROBUST_SEEDS:
    print(f"Longer-window repeated-seed ablation: seed {seed}")

    candidate_rules, selected_rules = mine_validated_rules(
        train_val_robust,
        robust_base_spec["feature_cols"],
        robust_base_spec["return_col"],
        rule_blocks=robust_base_spec.get("rule_blocks"),
        max_pair_features=robust_base_spec.get("max_pair_features", 36),
        max_rules_per_label=robust_base_spec.get("max_rules_per_label", 12),
    )

    robust_ltn_spec = robust_base_spec.copy()
    robust_ltn_spec["selected_rules"] = selected_rules.copy()

    robust_seed_rule_rows.append({
        "seed": seed,
        "model": "LTN, article-event + context interaction",
        "rule_blocks": ", ".join(robust_base_spec.get("rule_blocks", [])),
        "n_candidate_rules": len(candidate_rules),
        "n_selected_rules": len(selected_rules),
        "validation_n_min": selected_rules["validation_n"].min() if len(selected_rules) else np.nan,
        "validation_n_median": selected_rules["validation_n"].median() if len(selected_rules) else np.nan,
        "validation_n_max": selected_rules["validation_n"].max() if len(selected_rules) else np.nan,
        "validation_accuracy_mean": selected_rules["validation_accuracy_pct"].mean() if len(selected_rules) else np.nan,
    })

    robust_ltn_bundle = train_variant(
        f"robust_ltn_qwen_context_interaction_seed_{seed}",
        robust_ltn_spec,
        train_val_robust,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    robust_ltn_scored = score_variant(robust_ltn_bundle, test_robust)

    robust_seed_rows.append({
        "seed": seed,
        "model": "LTN, article-event + context interaction",
        **summary_row(
            "LTN, article-event + context interaction",
            "robust_heldout_test",
            robust_ltn_scored,
            bundle=robust_ltn_bundle,
        ),
    })

    robust_no_logic_bundle = train_variant(
        f"robust_no_logic_qwen_context_interaction_seed_{seed}",
        robust_base_spec,
        train_val_robust,
        use_logic=False,
        logic_weight=0.0,
        seed=seed,
        mine_rules=False,
    )

    robust_no_logic_scored = score_variant(robust_no_logic_bundle, test_robust)

    robust_seed_rows.append({
        "seed": seed,
        "model": "Neural classifier, Qwen + context interaction, no logic",
        **summary_row(
            "Neural classifier, Qwen + context interaction, no logic",
            "robust_heldout_test",
            robust_no_logic_scored,
            bundle=robust_no_logic_bundle,
        ),
    })


robust_seed_results = pd.DataFrame(robust_seed_rows)
robust_seed_rule_summary = pd.DataFrame(robust_seed_rule_rows)

robust_seed_summary = (
    robust_seed_results
    .groupby("model")
    .agg(
        runs=("seed", "nunique"),
        accuracy_mean=("accuracy_pct", "mean"),
        accuracy_std=("accuracy_pct", "std"),
        aligned_mean_return_mean=("aligned_mean_pct", "mean"),
        aligned_mean_return_std=("aligned_mean_pct", "std"),
        confidence_margin_mean=("mean_confidence_margin", "mean"),
        confidence_margin_std=("mean_confidence_margin", "std"),
        selected_rules_mean=("n_selected_rules", "mean"),
        selected_rules_min=("n_selected_rules", "min"),
        selected_rules_max=("n_selected_rules", "max"),
    )
    .reset_index()
)

display(robust_seed_rule_summary)
display(robust_seed_results)
display(robust_seed_summary)


Longer-window repeated-seed ablation: seed 1
Longer-window repeated-seed ablation: seed 2
Longer-window repeated-seed ablation: seed 3
Longer-window repeated-seed ablation: seed 4
Longer-window repeated-seed ablation: seed 5
Longer-window repeated-seed ablation: seed 7
Longer-window repeated-seed ablation: seed 11
Longer-window repeated-seed ablation: seed 13
Longer-window repeated-seed ablation: seed 17
Longer-window repeated-seed ablation: seed 19


,seed,model,rule_blocks,n_candidate_rules,n_selected_rules,validation_n_min,validation_n_median,validation_n_max,validation_accuracy_mean
0,1,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1290,15,5,16.0,78,80.694979
1,2,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1290,15,5,16.0,78,80.694979
2,3,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1290,15,5,16.0,78,80.694979
3,4,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1290,15,5,16.0,78,80.694979
4,5,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1290,15,5,16.0,78,80.694979
5,7,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1290,15,5,16.0,78,80.694979
6,11,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1290,15,5,16.0,78,80.694979
7,13,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1290,15,5,16.0,78,80.694979
8,17,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1290,15,5,16.0,78,80.694979
9,19,"LTN, article-event + context interaction","qwen_semantic, market_context, qwen_context_in...",1290,15,5,16.0,78,80.694979


,seed,model,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,mean_max_class_score,n_independent_clusters,aligned_mean_ci_low_pct,aligned_mean_ci_high_pct,aligned_bootstrap_p_value,use_logic,logic_weight,n_selected_rules
0,1,"LTN, article-event + context interaction","LTN, article-event + context interaction",robust_heldout_test,1633,40.845070,2.72948,0.191232,0.866718,44.396816,47.336191,8.266993,0.835888,0.914812,204,-0.582410,1.028668,0.3314,True,0.3,15
1,1,"Neural classifier, Qwen + context interaction,...","Neural classifier, Qwen + context interaction,...",robust_heldout_test,1633,45.192897,2.72948,0.429822,0.039798,46.662584,45.499081,7.838334,0.829148,0.911399,198,-0.346689,1.228708,0.1546,False,0.0,0
2,2,"LTN, article-event + context interaction","LTN, article-event + context interaction",robust_heldout_test,1633,43.784446,2.72948,0.408903,0.448946,47.581139,45.376607,7.042254,0.862338,0.928113,204,-0.324629,1.229817,0.1496,True,0.3,15
3,2,"Neural classifier, Qwen + context interaction,...","Neural classifier, Qwen + context interaction,...",robust_heldout_test,1633,44.825475,2.72948,0.262513,0.370307,47.397428,47.030006,5.572566,0.844331,0.919052,204,-0.468723,1.035595,0.2590,False,0.0,0
4,3,"LTN, article-event + context interaction","LTN, article-event + context interaction",robust_heldout_test,1633,41.641151,2.72948,0.062347,0.989403,47.152480,48.377220,4.470300,0.859511,0.927075,204,-0.671966,0.843569,0.4534,True,0.3,15
5,3,"Neural classifier, Qwen + context interaction,...","Neural classifier, Qwen + context interaction,...",robust_heldout_test,1633,43.355787,2.72948,0.284890,0.138795,45.192897,42.131047,12.676056,0.850015,0.921660,198,-0.449430,1.032763,0.2424,False,0.0,0
6,4,"LTN, article-event + context interaction","LTN, article-event + context interaction",robust_heldout_test,1633,45.315370,2.72948,0.326229,0.500000,45.866503,50.459277,3.674219,0.871024,0.933080,204,-0.459150,1.152474,0.2206,True,0.3,15
7,4,"Neural classifier, Qwen + context interaction,...","Neural classifier, Qwen + context interaction,...",robust_heldout_test,1633,47.152480,2.72948,0.587186,0.002595,47.213717,45.988977,6.797306,0.828484,0.908574,206,-0.201587,1.409892,0.0804,False,0.0,0
8,5,"LTN, article-event + context interaction","LTN, article-event + context interaction",robust_heldout_test,1633,48.560931,2.72948,0.723428,0.000025,51.684017,41.212492,7.103491,0.828202,0.909445,200,0.000130,1.497545,0.0250,True,0.3,15
9,5,"Neural classifier, Qwen + context interaction,...","Neural classifier, Qwen + context interaction,...",robust_heldout_test,1633,42.253521,2.72948,-0.083542,0.866878,42.437232,49.173301,8.389467,0.830808,0.911852,204,-0.709939,0.566283,0.6174,False,0.0,0


,model,runs,accuracy_mean,accuracy_std,aligned_mean_return_mean,aligned_mean_return_std,confidence_margin_mean,confidence_margin_std,selected_rules_mean,selected_rules_min,selected_rules_max
0,"LTN, article-event + context interaction",10,44.923454,2.692447,0.397378,0.243405,0.853742,0.018404,15.0,15,15
1,"Neural classifier, Qwen + context interaction,...",10,44.170239,1.935203,0.253421,0.219146,0.840050,0.009006,0.0,0,0


**Reading this table:** on the robustness split, LTN (44.9% $\pm$ 2.7% accuracy) is marginally ahead of no-logic (44.2% $\pm$ 1.9%) -- the opposite direction from the primary split's repeated-seed ablation (Section 11), where no-logic was marginally ahead. In both cases the difference is well within one standard deviation. See Section 16.

## 16. Results Synthesis And Discussion

The sections above report accuracy and aligned-return tables in isolation. This section states plainly what the combined evidence supports, and what it does not.

### Does the article-event / LLM-semantic signal add value over price context alone?

Not on this evidence. On the primary test window (Section 9), `full_ltn_context_only` -- price and market context only, no LLM-derived event features -- is both the most accurate variant (47.7%) and the only one with a statistically significant aligned return (p = 0.017). `full_ltn_qwen_context_interaction`, the variant carrying the article-event and interaction features that this notebook's title and framing are built around, is the least accurate of the four (42.6%) and is not significant (p = 0.21). The rule-generalisation check (Section 11) reinforces this: LLM/news-conditioned rules collapse from validation to test far more often and more severely than price-context rules.

### Is the LTN logic penalty earning its added complexity?

No robust evidence that it is. The repeated-seed ablation on the primary split (Section 11) shows LTN and an equivalent no-logic neural network as statistically indistinguishable (42.4% vs 42.5% accuracy; aligned mean return identical to two decimal places). The same ablation on the robustness split (Section 15) shows LTN marginally ahead instead -- the effect switches sign between windows while remaining within one standard deviation of zero difference in both cases. Taken together, this is a consistent null result, not an inconclusive one: across every ablation run in this project, on two different splits, the logic-rule component has not produced a repeatable improvement over an equivalent plain neural network.

### How should the instability between windows be weighed?

The single most important cross-check in this notebook is the reversal between Section 9 and Section 13. On the primary test window, `full_ltn_context_only` wins and `full_ltn_qwen_context_interaction` is both the weakest and non-significant variant. On the robustness window, that ranking inverts: `full_ltn_qwen_context_interaction` becomes the strongest and only significant variant, and `full_ltn_context_only` becomes negative and non-significant. Neither window is more "correct" than the other -- both are legitimate chronological, leakage-safe held-out splits. The reversal itself is the finding: model ranking on this dataset is highly sensitive to which window is used for evaluation, which means any single-window claim about which variant is "best" would not have survived a second check. This is exactly the kind of overfitting risk a robustness section exists to surface, and it did.

### What does generalise?

Two things, with different levels of confidence. First, and most tentatively: the pure price-context features are consistently the more reliable signal source across both windows and in the rule-generalisation check, even though which *variant* wins flips between windows. Second, and more concretely: the high-confidence (0.90 quantile) selective bullish signal in Section 12 is significant in every one of 10 seeds on the primary test window (median p = 0.000004, aligned mean return 2.62%), which is a real, repeatable effect within that window. It should not yet be read as a stable property of the model in general -- the validation-only calibration grid in the same section predicted the opposite direction (a bearish, not bullish, advantage), so this specific asymmetry may be particular to the Mar-Jun 2026 test period rather than a generalisable calibration property.

### Honest summary

This notebook does not provide robust evidence that fusing LLM-derived event semantics with a logic-rule layer improves MAG7 return classification over a simpler price-context model. The strongest, most consistent signal in the whole pipeline comes from price/market context alone, and the added complexity of the article-event features and the LTN rule layer does not reliably pay for itself across the two independent test windows examined. The main positive finding worth further investigation is the high-confidence selective bullish signal, which is real and seed-stable within the primary test window but not yet validated against the robustness window and not predicted by the model's own validation-time calibration. Reported honestly, this is a legitimate negative/mixed result obtained through a leakage-safe, statistically valid pipeline -- not a demonstration that the proposed neuro-symbolic approach works.